In [ ]:
# 0. Imports and Google Drive
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from difflib import get_close_matches
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid", context="notebook")

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": __import__("scipy").__version__,
    "colab": IN_COLAB,
})

In [ ]:
# 1. One-line compartment selector
TARGET_COMPARTMENT = "non_immune"  # immune_course | immune_fine | non_immune

MODE = "AUDIT_ONLY"
AUDIT_VERSION = "bayesprism-initial-issue-verification-v1.1"
PROJECT_REPOSITORY_COMMIT = "fe14ec848bfeac469528dc87c5aea8aacb14a87c"
RUN_MODEL = False
RDS_PATH_OVERRIDE = None
AUTO_INSTALL_R_STACK = True
HASH_LARGE_RDS = False
SAVE_FULL_SCORE_TABLE = True

BASE = Path("/content/drive/MyDrive/new_dysfunction")
BAYESPRISM_ROOT = Path("/content/drive/MyDrive/BayesPrism")
FINAL_SIGNATURE_JSON = BASE / "signature/Final_Refined_Signatures_Gemini.json"
MAPPING_CSV = BASE / "signature/mapping_all_final.csv"
CLEAN_ROOT = BASE / "cleaned_expression_3decon"
BMI_CSV = BAYESPRISM_ROOT / "data/sample_id_with_BMI.csv"
CLINICAL_CSV = Path("/content/drive/MyDrive/BHM_simulation_results/cptac_complete_clinical.csv")
AUDIT_ROOT = BASE / "bayesprism_initial_issue_verification_v1"

EXPECTED_COHORT_COUNTS = {"normal_weight": 52, "overweight": 57, "obese": 18}
EXPECTED_PATIENTS = 127
MAX_GENE_MISSING_FRACTION = 0.20
MIN_SIGNATURE_GENES = 4
MIN_GENE_COVERAGE = 0.50
NEAR_OVERLAP_THRESHOLD = 0.50
EMPIRICAL_CORRELATION_THRESHOLD = 0.90
LOW_THETA_THRESHOLD = 0.005
HIGH_THETA_CV_THRESHOLD = 1.0
MAX_RDS_TOTAL_RELATIVE_ERROR = 1e-6
CLIP_CAPS = (3.0, 5.0)

COMPARTMENTS = {
    "immune_course": {
        "module": "immune_course",
        "level": "type_level",
        "theta_kind": "type_final",
        "theta_file": "theta_type_final.csv",
        "theta_cv_file": "theta_type_final_cv.csv",
        "z_totals_file": "z_totals_type.csv",
        "rds_candidates": [
            BAYESPRISM_ROOT / "results_annotation/bayesprism_results.rds",
            Path("/content/bayesprism_immune_course.rds"),
        ],
        "rds_basenames": ["bayesprism_results.rds", "bayesprism_immune_course.rds"],
    },
    "immune_fine": {
        "module": "immune_fine",
        "level": "state_level",
        "theta_kind": "state_initial",
        "theta_file": "theta_state_initial.csv",
        "theta_cv_file": "theta_state_initial_cv.csv",
        "z_totals_file": "z_totals_state.csv",
        "rds_candidates": [
            BAYESPRISM_ROOT / "results_annotation/bayesprism_results_FINE_TAM.rds",
            Path("/content/bayesprism_immune_fine.rds"),
        ],
        "rds_basenames": ["bayesprism_results_FINE_TAM.rds", "bayesprism_immune_fine.rds"],
    },
    "non_immune": {
        "module": "non_immune",
        "level": "type_level",
        "theta_kind": "type_final",
        "theta_file": "theta_type_final.csv",
        "theta_cv_file": "theta_type_final_cv.csv",
        "z_totals_file": "z_totals_type.csv",
        "rds_candidates": [
            BAYESPRISM_ROOT / "improved_deconvolution_results/bayesprism_object.rds",
            Path("/content/bayesprism_non_immune.rds"),
        ],
        "rds_basenames": ["bayesprism_object.rds", "bayesprism_non_immune.rds"],
    },
}



THETA_NAME_OVERRIDES = {}
Z_NAME_OVERRIDES = {}

assert MODE == "AUDIT_ONLY"
assert TARGET_COMPARTMENT in COMPARTMENTS
assert RUN_MODEL is False
CFG = COMPARTMENTS[TARGET_COMPARTMENT]
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AUDIT_DIR = AUDIT_ROOT / TARGET_COMPARTMENT / RUN_STAMP
THETA_DIR = AUDIT_DIR / "theta_from_rds"
PLOT_DIR = AUDIT_DIR / "plots"
for directory in (AUDIT_DIR, THETA_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=False)

print("Target:", TARGET_COMPARTMENT, CFG["module"], CFG["level"])
print("Audit output:", AUDIT_DIR)

In [ ]:
# 2. Shared helpers
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_token(value) -> str:
    text = str(value).strip().upper().replace("Γ", "GAMMA").replace("Δ", "DELTA")
    return re.sub(r"[^A-Z0-9]+", "_", text).strip("_")


def to_base_id(value):
    if pd.isna(value):
        return np.nan
    text = str(value).split(";", 1)[0].strip().upper()
    match = re.search(r"(C3[NL])[-_]?(\d{5})", text)
    return f"{match.group(1)}-{match.group(2)}" if match else text


def bmi_group(value):
    if pd.isna(value):
        return np.nan
    value = float(value)
    if value < 18.5:
        return "underweight"
    if value < 25.0:
        return "normal_weight"
    if value < 30.0:
        return "overweight"
    return "obese"


def normalized_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    names = [canonical_token(column).lower() for column in out.columns]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"Columns collide after normalization: {duplicates}")
    out.columns = names
    return out


def find_column(frame: pd.DataFrame, candidates):
    for candidate in candidates:
        key = canonical_token(candidate).lower()
        if key in frame.columns:
            return key
    return None


def atomic_to_csv(frame: pd.DataFrame, path: Path, **kwargs):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    frame.to_csv(tmp, index=False, **kwargs)
    tmp.replace(path)


def atomic_write_text(text: str, path: Path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def safe_spearman(x, y):
    pair = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(pair) < 4 or pair["x"].nunique() < 2 or pair["y"].nunique() < 2:
        return np.nan, np.nan, len(pair)
    result = stats.spearmanr(pair["x"], pair["y"])
    return float(result.statistic), float(result.pvalue), len(pair)


def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full(p.shape, np.nan)
    valid = np.isfinite(p)
    pv = p[valid]
    if not len(pv):
        return q
    order = np.argsort(pv)
    ranked = pv[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    restored = np.empty_like(adjusted)
    restored[order] = np.minimum(adjusted, 1.0)
    q[valid] = restored
    return q


def split_genes(value):
    if value is None:
        return []
    if isinstance(value, str):
        values = re.split(r"[,;|\s]+", value.strip())
    else:
        try:
            values = list(value)
        except TypeError:
            values = [value]
    return sorted({str(gene).strip().upper() for gene in values if str(gene).strip()})


def signed_genes(payload):
    if not isinstance(payload, dict):
        return split_genes(payload), []
    positive, negative = [], []
    for key in ("positive", "pos", "up", "up_genes", "UP", "upregulated"):
        if key in payload:
            positive.extend(split_genes(payload[key]))
    for key in ("negative", "neg", "down", "down_genes", "DN", "downregulated"):
        if key in payload:
            negative.extend(split_genes(payload[key]))
    return sorted(set(positive)), sorted(set(negative))


def connected_component_table(nodes, edges, *, prefix):
    """Deterministic connected components; singleton nodes are retained."""
    nodes = sorted(set(nodes))
    parent = {node: node for node in nodes}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left, root_right = find(left), find(right)
        if root_left == root_right:
            return
        keep, merge = sorted([root_left, root_right])
        parent[merge] = keep

    for left, right in edges:
        if left in parent and right in parent:
            union(left, right)

    groups = {}
    for node in nodes:
        groups.setdefault(find(node), []).append(node)
    ordered = sorted((sorted(members) for members in groups.values()), key=lambda members: members[0])
    rows = []
    for index, members in enumerate(ordered, start=1):
        cluster_id = f"{prefix}_{index:04d}"
        for member in members:
            rows.append({
                "cluster_id": cluster_id,
                "member": member,
                "cluster_size": len(members),
                "is_singleton": len(members) == 1,
            })
    return pd.DataFrame(rows)


# Tiny non-data-dependent self-test
assert to_base_id("C3L-00589-01;C3L-00589-02") == "C3L-00589"
assert canonical_token("γδ T cells") == "GAMMADELTA_T_CELLS"
assert bmi_group(29.9) == "overweight" and bmi_group(30) == "obese"
assert np.allclose(bh_adjust([0.01, 0.04, 0.03]), [0.03, 0.04, 0.04])
_cc_test = connected_component_table(["a", "b", "c"], [("a", "b")], prefix="test")
assert sorted(_cc_test["cluster_size"]) == [1, 2, 2]
print("Helper self-test: PASS")

In [ ]:
# 3. Resolve required files and the BayesPrism RDS
def resolve_rds_path():
    candidates = []
    if RDS_PATH_OVERRIDE:
        override = Path(RDS_PATH_OVERRIDE)
        if not override.exists():
            raise FileNotFoundError(f"RDS_PATH_OVERRIDE does not exist: {override}")
        candidates.append(override)
    candidates.extend(Path(path) for path in CFG["rds_candidates"])

    if not any(path.exists() for path in candidates) and BAYESPRISM_ROOT.exists():
        for basename in CFG["rds_basenames"]:
            candidates.extend(BAYESPRISM_ROOT.rglob(basename))

    unique = []
    seen = set()
    for path in candidates:
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)

    inventory = pd.DataFrame([
        {
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
            "mtime_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat() if path.exists() else None,
            "is_override": bool(RDS_PATH_OVERRIDE and str(path) == str(Path(RDS_PATH_OVERRIDE))),
        }
        for path in unique
    ])
    found = [path for path in unique if path.exists()]
    if not found:
        display(inventory)
        raise FileNotFoundError(
            "No BayesPrism RDS found. Upload it or set RDS_PATH_OVERRIDE to its exact Drive path."
        )
    selected = Path(RDS_PATH_OVERRIDE) if RDS_PATH_OVERRIDE else found[0]
    inventory["selected"] = inventory["path"].eq(str(selected))
    return selected, inventory


RDS_PATH, rds_inventory = resolve_rds_path()
required = {
    "signature_json": FINAL_SIGNATURE_JSON,
    "mapping_csv": MAPPING_CSV,
    "clean_compartment_dir": CLEAN_ROOT / CFG["module"] / CFG["level"],
    "bmi_csv": BMI_CSV,
    "clinical_csv": CLINICAL_CSV,
    "bayesprism_rds": RDS_PATH,
}
path_rows = []
for role, path in required.items():
    exists = path.exists()
    path_rows.append({
        "role": role,
        "path": str(path),
        "exists": exists,
        "size_bytes": path.stat().st_size if exists and path.is_file() else np.nan,
        "sha256": (
            sha256_file(path)
            if exists and path.is_file() and (role != "bayesprism_rds" or HASH_LARGE_RDS)
            else None
        ),
    })
path_audit = pd.DataFrame(path_rows)
atomic_to_csv(rds_inventory, AUDIT_DIR / "rds_candidate_inventory.csv")
atomic_to_csv(path_audit, AUDIT_DIR / "path_preflight.csv")
display(path_audit)
if not path_audit["exists"].all():
    raise FileNotFoundError("One or more required inputs are missing; inspect path_preflight.csv")

print("Resolved RDS:", RDS_PATH)

In [ ]:
# 4. Ensure R and BayesPrism are available
def run_checked(command, *, label, timeout=None):
    started = time.time()
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    print(f"{label}: returncode={result.returncode}; elapsed={time.time()-started:.1f}s")
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-8000:])
        raise RuntimeError(f"{label} failed")
    return result


if shutil.which("Rscript") is None:
    if not (IN_COLAB and AUTO_INSTALL_R_STACK):
        raise RuntimeError("Rscript is missing. Enable AUTO_INSTALL_R_STACK in Colab or use a runtime with R.")
    run_checked(["apt-get", "update", "-qq"], label="apt update", timeout=1200)
    run_checked(
        [
            "apt-get", "install", "-y", "-qq", "r-base", "r-base-dev",
            "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev",
            "libgsl-dev", "libfftw3-dev",
        ],
        label="R installation",
        timeout=2400,
    )

check = subprocess.run(
    ["Rscript", "-e", "quit(status=ifelse(requireNamespace('BayesPrism', quietly=TRUE),0,1))"],
    text=True,
    capture_output=True,
)
if check.returncode != 0:
    if not AUTO_INSTALL_R_STACK:
        raise RuntimeError("BayesPrism is not installed; set AUTO_INSTALL_R_STACK=True and rerun this cell.")
    install_script = r"""
    options(repos=c(CRAN="https://cloud.r-project.org"))
    if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
    options(repos=BiocManager::repositories())
    BiocManager::install(c("BiocParallel", "scran"), ask=FALSE, update=FALSE, Ncpus=2)
    install.packages(c("snowfall", "NMF", "gplots", "remotes", "Rcpp", "RcppArmadillo"), Ncpus=2)
    remotes::install_github("Danko-Lab/BayesPrism", subdir="BayesPrism", dependencies=FALSE, upgrade="never")
    stopifnot(requireNamespace("BayesPrism", quietly=TRUE))
    """
    install_path = AUDIT_DIR / "install_bayesprism.R"
    install_path.write_text(install_script, encoding="utf-8")
    run_checked(["Rscript", str(install_path)], label="BayesPrism installation", timeout=7200)

version_result = run_checked(
    ["Rscript", "-e", "cat(as.character(packageVersion('BayesPrism')))"],
    label="BayesPrism check",
    timeout=120,
)
BAYESPRISM_VERSION = version_result.stdout.strip()

In [ ]:
# 5. Write and run a strict slot-level R extractor
R_EXTRACTOR = r"""
args <- commandArgs(trailingOnly=TRUE)
if (length(args) != 2) stop("Expected: RDS_PATH OUTPUT_DIR")
rds_path <- args[[1]]
out_dir <- args[[2]]
dir.create(out_dir, recursive=TRUE, showWarnings=FALSE)
suppressPackageStartupMessages(library(BayesPrism))

has_slot <- function(object, name) isS4(object) && name %in% slotNames(object)
nested_slot <- function(object, outer, inner) {
  if (!has_slot(object, outer)) return(NULL)
  child <- slot(object, outer)
  if (!has_slot(child, inner)) return(NULL)
  slot(child, inner)
}
to_base_id <- function(x) {
  s <- toupper(trimws(as.character(x)))
  first <- sub(";.*$", "", s)
  m <- regexec("(C3[NL])[-_]?(\\d{5})", first, perl=TRUE)
  pieces <- regmatches(first, m)
  vapply(pieces, function(z) if (length(z)==3) paste0(z[[2]], "-", z[[3]]) else NA_character_, character(1))
}

bp <- readRDS(rds_path)
if (!isS4(bp) || !is(bp, "BayesPrism")) stop("RDS is not a BayesPrism S4 object")

expected_samples <- NULL
if (has_slot(bp, "prism")) {
  prism <- slot(bp, "prism")
  if (has_slot(prism, "mixture")) expected_samples <- rownames(slot(prism, "mixture"))
}

orientation_score <- function(mat, expected) {
  rn <- rownames(mat)
  overlap <- 0
  if (!is.null(expected) && !is.null(rn)) overlap <- mean(expected %in% rn)
  id_like <- if (is.null(rn)) 0 else mean(grepl("C3[NL][-_]?[0-9]{5}", toupper(rn)))
  sums <- rowSums(mat, na.rm=TRUE)
  mass <- if (length(sums)) median(abs(sums - 1), na.rm=TRUE) else Inf
  if (!is.finite(mass)) mass <- 1e6
  100 * overlap + 10 * id_like - mass
}
orient_theta <- function(object, expected) {
  mat <- as.matrix(object)
  candidates <- list(as_is=mat, transposed=t(mat))
  scores <- vapply(candidates, orientation_score, numeric(1), expected=expected)
  choice <- names(which.max(scores))[[1]]
  out <- candidates[[choice]]
  if ((is.null(rownames(out)) || any(rownames(out)=="")) && !is.null(expected) && nrow(out)==length(expected)) {
    rownames(out) <- expected
  }
  list(matrix=out, orientation=choice, score=scores[[choice]])
}
write_wide <- function(mat, path) {
  ids <- rownames(mat)
  if (is.null(ids)) ids <- paste0("Sample_", seq_len(nrow(mat)))
  frame <- data.frame(sample_id=ids, base_sample_id=to_base_id(ids), as.data.frame(mat, check.names=FALSE), check.names=FALSE)
  write.csv(frame, path, row.names=FALSE)
}

manifest <- list()
add_manifest <- function(name, source_slot, available, dims="", orientation="", row_sum_min=NA, row_sum_median=NA, row_sum_max=NA, note="") {
  manifest[[length(manifest)+1]] <<- data.frame(
    artifact=name, source_slot=source_slot, available=available, dimensions=dims,
    orientation=orientation, row_sum_min=row_sum_min, row_sum_median=row_sum_median,
    row_sum_max=row_sum_max, note=note, stringsAsFactors=FALSE
  )
}

theta_specs <- list(
  theta_state_initial=list(value=nested_slot(bp, "posterior.initial.cellState", "theta"), source="posterior.initial.cellState@theta"),
  theta_state_initial_cv=list(value=nested_slot(bp, "posterior.initial.cellState", "theta.cv"), source="posterior.initial.cellState@theta.cv"),
  theta_type_initial=list(value=nested_slot(bp, "posterior.initial.cellType", "theta"), source="posterior.initial.cellType@theta"),
  theta_type_initial_cv=list(value=nested_slot(bp, "posterior.initial.cellType", "theta.cv"), source="posterior.initial.cellType@theta.cv"),
  theta_type_final=list(value=nested_slot(bp, "posterior.theta_f", "theta"), source="posterior.theta_f@theta"),
  theta_type_final_cv=list(value=nested_slot(bp, "posterior.theta_f", "theta.cv"), source="posterior.theta_f@theta.cv")
)

for (name in names(theta_specs)) {
  spec <- theta_specs[[name]]
  if (is.null(spec$value)) {
    add_manifest(name, spec$source, FALSE, note="slot unavailable")
    next
  }
  oriented <- orient_theta(spec$value, expected_samples)
  mat <- oriented$matrix
  sums <- rowSums(mat, na.rm=TRUE)
  write_wide(mat, file.path(out_dir, paste0(name, ".csv")))
  add_manifest(
    name, spec$source, TRUE, paste(dim(mat), collapse=" x "), oriented$orientation,
    min(sums, na.rm=TRUE), median(sums, na.rm=TRUE), max(sums, na.rm=TRUE),
    if (grepl("_cv$", name)) "Row sums are not expected to equal one for CV" else ""
  )
}

write_z_totals <- function(z, level, source) {
  artifact <- paste0("z_totals_", level)
  if (is.null(z) || length(dim(z)) != 3) {
    add_manifest(artifact, source, FALSE, note="3-D Z slot unavailable")
    return(NULL)
  }
  dims <- dim(z)
  dn <- dimnames(z)
  if (!is.null(expected_samples) && dims[[1]] != length(expected_samples)) {
    add_manifest(artifact, source, FALSE, paste(dims, collapse=" x "), note="first Z dimension does not match mixture samples")
    return(NULL)
  }
  sample_names <- dn[[1]]
  if (is.null(sample_names)) sample_names <- if (!is.null(expected_samples)) expected_samples else paste0("Sample_", seq_len(dims[[1]]))
  gene_names <- dn[[2]]
  if (is.null(gene_names)) gene_names <- paste0("gene_", seq_len(dims[[2]]))
  component_names <- dn[[3]]
  if (is.null(component_names)) component_names <- paste0(level, "_", seq_len(dims[[3]]))
  writeLines(as.character(gene_names), file.path(out_dir, paste0("z_genes_", level, ".txt")))
  writeLines(as.character(component_names), file.path(out_dir, paste0("z_components_", level, ".txt")))
  totals <- matrix(NA_real_, nrow=dims[[1]], ncol=dims[[3]], dimnames=list(sample_names, component_names))
  for (index in seq_len(dims[[3]])) {
    slice <- z[,,index, drop=TRUE]
    if (is.null(dim(slice))) slice <- matrix(slice, nrow=dims[[1]])
    totals[,index] <- rowSums(slice, na.rm=TRUE)
  }
  write_wide(totals, file.path(out_dir, paste0(artifact, ".csv")))
  add_manifest(artifact, source, TRUE, paste(dims, collapse=" x "), "samples x genes x components", note="CSV contains component totals only")
}

write_z_totals(nested_slot(bp, "posterior.initial.cellState", "Z"), "state", "posterior.initial.cellState@Z")
write_z_totals(nested_slot(bp, "posterior.initial.cellType", "Z"), "type", "posterior.initial.cellType@Z")

manifest_df <- do.call(rbind, manifest)
write.csv(manifest_df, file.path(out_dir, "bp_slot_manifest.csv"), row.names=FALSE)
object_info <- data.frame(
  rds_path=rds_path,
  bayesprism_version=as.character(packageVersion("BayesPrism")),
  object_class=paste(class(bp), collapse=";"),
  object_slots=paste(slotNames(bp), collapse=";"),
  expected_samples=ifelse(is.null(expected_samples), NA_integer_, length(expected_samples)),
  stringsAsFactors=FALSE
)
write.csv(object_info, file.path(out_dir, "bp_object_info.csv"), row.names=FALSE)
cat("BayesPrism slot extraction complete\n")
print(manifest_df)
"""

r_script_path = AUDIT_DIR / "extract_bayesprism_slots.R"
r_script_path.write_text(R_EXTRACTOR, encoding="utf-8")
extraction = run_checked(
    ["Rscript", str(r_script_path), str(RDS_PATH), str(THETA_DIR)],
    label="BayesPrism slot extraction",
    timeout=7200,
)
atomic_write_text(extraction.stdout, AUDIT_DIR / "r_extraction_stdout.log")
atomic_write_text(extraction.stderr, AUDIT_DIR / "r_extraction_stderr.log")

bp_slot_manifest = pd.read_csv(THETA_DIR / "bp_slot_manifest.csv")
bp_object_info = pd.read_csv(THETA_DIR / "bp_object_info.csv")
display(bp_object_info)
display(bp_slot_manifest)

In [ ]:
# 6. Load, validate, and compare theta with normalized Z totals
ID_COLUMNS = {"sample_id", "base_sample_id"}


def read_wide_numeric(path: Path):
    frame = pd.read_csv(path)
    if "base_sample_id" not in frame:
        raise ValueError(f"Missing base_sample_id in {path}")
    frame["base_sample_id"] = frame["base_sample_id"].map(to_base_id)
    numeric_columns = [column for column in frame.columns if column not in ID_COLUMNS]
    values = frame[numeric_columns].apply(pd.to_numeric, errors="coerce")
    values.index = frame["base_sample_id"]
    values = values.groupby(level=0).median()
    return values


theta_path = THETA_DIR / CFG["theta_file"]
theta_cv_path = THETA_DIR / CFG["theta_cv_file"]
if not theta_path.exists():
    raise FileNotFoundError(
        f"Required {CFG['theta_kind']} theta was not found in the RDS: {theta_path.name}"
    )

theta_target = read_wide_numeric(theta_path)
theta_cv_target = read_wide_numeric(theta_cv_path) if theta_cv_path.exists() else pd.DataFrame(index=theta_target.index)
z_totals_target_path = THETA_DIR / CFG["z_totals_file"]
z_level = "state" if CFG["level"] == "state_level" else "type"
z_genes_target_path = THETA_DIR / f"z_genes_{z_level}.txt"
if not z_totals_target_path.exists() or not z_genes_target_path.exists():
    raise FileNotFoundError(
        "The target RDS does not expose the initial Z array needed to verify the cleaned matrices: "
        f"{z_totals_target_path.name}, {z_genes_target_path.name}"
    )
z_totals_target = read_wide_numeric(z_totals_target_path)
rds_z_genes = [
    line.strip().upper()
    for line in z_genes_target_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
rds_z_gene_set = set(rds_z_genes)
if not rds_z_genes or len(rds_z_gene_set) != len(rds_z_genes):
    raise ValueError("RDS Z gene names are empty or duplicate after uppercase normalization")

theta_row_sums = theta_target.sum(axis=1, min_count=1)
theta_validation = {
    "theta_kind": CFG["theta_kind"],
    "n_patients": int(theta_target.shape[0]),
    "n_components": int(theta_target.shape[1]),
    "negative_values": int((theta_target < -1e-10).sum().sum()),
    "values_above_one": int((theta_target > 1 + 1e-10).sum().sum()),
    "nonfinite_values": int((~np.isfinite(theta_target.to_numpy(dtype=float))).sum()),
    "row_sum_min": float(theta_row_sums.min()),
    "row_sum_median": float(theta_row_sums.median()),
    "row_sum_max": float(theta_row_sums.max()),
    "fraction_rows_within_1e_6": float(np.mean(np.isclose(theta_row_sums, 1.0, atol=1e-6))),
    "fraction_rows_within_1e_3": float(np.mean(np.isclose(theta_row_sums, 1.0, atol=1e-3))),
}
atomic_write_text(json.dumps(theta_validation, indent=2), AUDIT_DIR / "theta_validation.json")

z_theta_rows = []
for level in ("state", "type"):
    z_path = THETA_DIR / f"z_totals_{level}.csv"
    initial_path = THETA_DIR / f"theta_{level}_initial.csv"
    if not (z_path.exists() and initial_path.exists()):
        continue
    z_totals = read_wide_numeric(z_path)
    z_fraction = z_totals.div(z_totals.sum(axis=1).replace(0, np.nan), axis=0)
    theta_initial = read_wide_numeric(initial_path)
    theta_by_key = {canonical_token(column): column for column in theta_initial.columns}
    z_by_key = {canonical_token(column): column for column in z_fraction.columns}
    for key in sorted(set(theta_by_key) & set(z_by_key)):
        t_col, z_col = theta_by_key[key], z_by_key[key]
        common = theta_initial.index.intersection(z_fraction.index)
        left = theta_initial.loc[common, t_col]
        right = z_fraction.loc[common, z_col]
        rho, pvalue, n = safe_spearman(left, right)
        z_theta_rows.append({
            "level": level,
            "component": t_col,
            "z_component": z_col,
            "n": n,
            "spearman_rho": rho,
            "p_value": pvalue,
            "max_abs_difference": float((left - right).abs().max()),
            "mean_abs_difference": float((left - right).abs().mean()),
        })
z_theta_consistency = pd.DataFrame(z_theta_rows)
atomic_to_csv(z_theta_consistency, AUDIT_DIR / "z_theta_consistency.csv")
display(pd.DataFrame([theta_validation]))
display(z_theta_consistency.head(20))

In [ ]:
# 7. Build the reviewed patient-level cohort
ID_CANDIDATES = [
    "base_sample_id", "sample_id", "sample", "case_submitter_id",
    "participant_id", "patient_id", "patientid", "subject_id", "case_id",
]
AGE_CANDIDATES = ["age", "age_at_diagnosis", "age_at_index", "age_at_initial_pathologic_diagnosis"]
SEX_CANDIDATES = ["sex", "gender", "biological_sex"]

bmi_raw = normalized_columns(pd.read_csv(BMI_CSV, low_memory=False))
bmi_id_col = find_column(bmi_raw, ID_CANDIDATES)
bmi_col = find_column(bmi_raw, ["bmi", "body_mass_index"])
if bmi_id_col is None or bmi_col is None:
    raise ValueError("BMI table must contain a recognized ID and BMI column")
bmi = pd.DataFrame({
    "base_sample_id": bmi_raw[bmi_id_col].map(to_base_id),
    "bmi": pd.to_numeric(bmi_raw[bmi_col], errors="coerce"),
}).dropna()
bmi_conflicts = bmi.groupby("base_sample_id")["bmi"].nunique(dropna=True)
if (bmi_conflicts > 1).any():
    raise ValueError("Conflicting BMI values exist for the same base patient")
bmi = bmi.groupby("base_sample_id", as_index=False)["bmi"].median()
bmi["bmi_group"] = bmi["bmi"].map(bmi_group)

clinical_raw = normalized_columns(pd.read_csv(CLINICAL_CSV, low_memory=False))
clinical_id_col = find_column(clinical_raw, ID_CANDIDATES)
age_col = find_column(clinical_raw, AGE_CANDIDATES)
sex_col = find_column(clinical_raw, SEX_CANDIDATES)
if clinical_id_col is None or age_col is None or sex_col is None:
    raise ValueError(
        f"Clinical ID/age/sex detection failed: id={clinical_id_col}, age={age_col}, sex={sex_col}"
    )
clinical = clinical_raw.copy()
clinical["base_sample_id"] = clinical[clinical_id_col].map(to_base_id)
clinical["age"] = pd.to_numeric(clinical[age_col], errors="coerce")
sex_text = clinical[sex_col].astype(str).str.strip().str.lower()
sex_map = {"female": -0.5, "f": -0.5, "male": 0.5, "m": 0.5}
clinical["sex_centered"] = sex_text.map(sex_map)

conflict_age = clinical.groupby("base_sample_id")["age"].nunique(dropna=True)
conflict_sex = clinical.groupby("base_sample_id")["sex_centered"].nunique(dropna=True)
if (conflict_age > 1).any() or (conflict_sex > 1).any():
    raise ValueError("Conflicting clinical values exist for at least one base patient")
clinical = clinical.groupby("base_sample_id", as_index=False).agg(
    age=("age", "median"), sex_centered=("sex_centered", "median")
)

cohort = bmi.merge(clinical, on="base_sample_id", how="left", validate="one_to_one")
cohort["bmi_z"] = (cohort["bmi"] - cohort["bmi"].mean()) / cohort["bmi"].std(ddof=1)
cohort["age_z"] = (cohort["age"] - cohort["age"].mean()) / cohort["age"].std(ddof=1)
cohort_counts = cohort.groupby("bmi_group")["base_sample_id"].nunique().to_dict()
cohort_counts = {str(key): int(value) for key, value in cohort_counts.items()}
theta_overlap = set(cohort["base_sample_id"]) & set(theta_target.index)
cohort_audit = {
    "n_bmi_patients": int(cohort["base_sample_id"].nunique()),
    "n_theta_patients": int(theta_target.index.nunique()),
    "n_bmi_theta_overlap": int(len(theta_overlap)),
    "cohort_counts": cohort_counts,
    "expected_counts": EXPECTED_COHORT_COUNTS,
    "age_missing": int(cohort["age"].isna().sum()),
    "sex_missing": int(cohort["sex_centered"].isna().sum()),
    "detected_columns": {"clinical_id": clinical_id_col, "age": age_col, "sex": sex_col},
}
atomic_to_csv(cohort, AUDIT_DIR / "analysis_cohort.csv")
atomic_write_text(json.dumps(cohort_audit, indent=2), AUDIT_DIR / "cohort_audit.json")
display(pd.DataFrame([cohort_audit]))

if len(cohort) != EXPECTED_PATIENTS or cohort_counts != EXPECTED_COHORT_COUNTS:
    raise ValueError("Cohort does not match the prespecified 127 patients and 52/57/18 groups")
if len(theta_overlap) != EXPECTED_PATIENTS:
    raise ValueError("Theta does not overlap all 127 analysis patients")
if cohort[["age", "sex_centered"]].isna().any().any():
    raise ValueError("Age or sex is missing in the 127-patient cohort")

In [ ]:
# 8. Resolve the compartment matrices and signatures
def resolve_mapping_columns(mapping: pd.DataFrame) -> pd.DataFrame:
    out = mapping.copy()
    if "matched_cell_types" not in out or "path" not in out:
        raise ValueError("mapping_all_final.csv needs path and matched_cell_types")
    if "module" not in out:
        out["module"] = out["path"].astype(str).str.extract(
            r"/(immune_course|immune_fine|non_immune)/", expand=False
        )
    if "level" not in out:
        out["level"] = out["path"].astype(str).str.extract(
            r"/(type_level|state_level)/", expand=False
        )
    if "file" not in out:
        out["file"] = out["path"].map(lambda value: Path(str(value)).name)
    return out


def resolve_clean_matrix(row) -> Path:
    filename = str(row["file"])
    matrix_name = re.sub(r"_expression\.csv$", "", filename, flags=re.IGNORECASE)
    base = CLEAN_ROOT / CFG["module"] / CFG["level"]
    parquet_path = base / f"{matrix_name}_CLEAN.parquet"
    csv_path = base / f"{matrix_name}_CLEAN.csv"
    have_parquet_engine = importlib.util.find_spec("pyarrow") is not None or importlib.util.find_spec("fastparquet") is not None
    candidates = [parquet_path, csv_path] if have_parquet_engine else [csv_path, parquet_path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No cleaned matrix found for {matrix_name} under {base}")


def load_signature_catalog(path: Path) -> pd.DataFrame:
    raw = json.loads(path.read_text(encoding="utf-8"))
    iterable = []
    if isinstance(raw, dict) and isinstance(raw.get("entries"), list):
        for entry in raw["entries"]:
            iterable.append((entry.get("cell_type"), entry.get("signature") or entry.get("name"), entry.get("genes")))
    elif isinstance(raw, dict):
        for cell_type, signatures in raw.items():
            if isinstance(signatures, dict):
                for signature, payload in signatures.items():
                    iterable.append((cell_type, signature, payload))
    else:
        raise ValueError("Unsupported signature JSON structure")

    rows = []
    for cell_type, signature, payload in iterable:
        if not cell_type or not signature:
            continue
        positive, negative = signed_genes(payload)
        all_genes = sorted(set(positive) | set(negative))
        if not all_genes:
            continue
        rows.append({
            "catalog_cell_type": str(cell_type).strip(),
            "signature": str(signature).strip(),
            "cell_key": canonical_token(cell_type),
            "signature_key": canonical_token(signature),
            "positive_genes": positive,
            "negative_genes": negative,
            "all_genes": all_genes,
            "n_genes": len(all_genes),
            "fingerprint": "POS:" + ",".join(positive) + "|NEG:" + ",".join(negative),
        })
    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError("No usable signatures in final catalogue")
    return result


def select_catalog(cell_type: str, catalog: pd.DataFrame) -> pd.DataFrame:
    exact = catalog[catalog["catalog_cell_type"] == cell_type]
    if not exact.empty:
        return exact.copy()
    candidates = catalog[catalog["cell_key"] == canonical_token(cell_type)]
    labels = candidates["catalog_cell_type"].unique()
    if len(labels) == 1:
        return candidates.copy()
    if len(labels) == 0:
        raise KeyError(f"No signatures found for mapped cell type: {cell_type}")
    raise ValueError(f"Ambiguous signature catalogue alias for {cell_type}: {sorted(labels)}")


mapping = resolve_mapping_columns(pd.read_csv(MAPPING_CSV, dtype=str).fillna(""))
mapping_target = mapping[(mapping["module"] == CFG["module"]) & (mapping["level"] == CFG["level"])].copy()
if mapping_target.empty:
    raise ValueError(f"No mapping rows for {CFG['module']}/{CFG['level']}")

target_rows = []
for row in mapping_target.to_dict(orient="records"):
    matrix_path = resolve_clean_matrix(row)
    for cell_type in [item.strip() for item in str(row["matched_cell_types"]).split(";") if item.strip()]:
        target_rows.append({
            "analysis_celltype": cell_type,
            "matrix_path": str(matrix_path),
            "mapping_source_path": str(row["path"]),
        })
matrix_targets = pd.DataFrame(target_rows).drop_duplicates(["analysis_celltype", "matrix_path"])
if matrix_targets.empty:
    raise ValueError("No compartment matrices/cell types resolved")

full_catalog = load_signature_catalog(FINAL_SIGNATURE_JSON)
selected_parts = []
for cell_type in matrix_targets["analysis_celltype"]:
    part = select_catalog(cell_type, full_catalog)
    part["analysis_celltype"] = cell_type
    selected_parts.append(part)
target_catalog_pre = pd.concat(selected_parts, ignore_index=True)

name_conflicts = []
for _, group in target_catalog_pre.groupby(["analysis_celltype", "signature_key"], sort=False):
    if group["fingerprint"].nunique() > 1:
        name_conflicts.append(group)
if name_conflicts:
    conflict_table = pd.concat(name_conflicts, ignore_index=True)
    atomic_to_csv(conflict_table.drop(columns=["positive_genes", "negative_genes", "all_genes"]), AUDIT_DIR / "signature_name_conflicts.csv")
    raise ValueError("A normalized signature name maps to different gene sets")

target_catalog_pre = target_catalog_pre.sort_values(["analysis_celltype", "signature", "fingerprint"]).reset_index(drop=True)
target_catalog_pre["keep_exact"] = ~target_catalog_pre.duplicated(
    ["analysis_celltype", "fingerprint"], keep="first"
)
exact_duplicate_drops = target_catalog_pre.loc[~target_catalog_pre["keep_exact"]].copy()
target_catalog = target_catalog_pre.loc[target_catalog_pre["keep_exact"]].copy()

near_rows = []
for cell_type, group in target_catalog.groupby("analysis_celltype", sort=True):
    items = list(group[["signature", "all_genes"]].itertuples(index=False, name=None))
    for left in range(len(items)):
        name_a, genes_a = items[left]
        set_a = set(genes_a)
        for right in range(left + 1, len(items)):
            name_b, genes_b = items[right]
            set_b = set(genes_b)
            denominator = min(len(set_a), len(set_b))
            overlap = len(set_a & set_b) / denominator if denominator else 0.0
            if overlap > NEAR_OVERLAP_THRESHOLD:
                near_rows.append({
                    "analysis_celltype": cell_type,
                    "signature_a": name_a,
                    "signature_b": name_b,
                    "overlap_coefficient": overlap,
                    "shared_genes": ";".join(sorted(set_a & set_b)),
                })
near_overlap_pairs = pd.DataFrame(near_rows)

overlap_cluster_parts = []
for cell_type, group in target_catalog.groupby("analysis_celltype", sort=True):
    if near_overlap_pairs.empty:
        edges = []
    else:
        local_pairs = near_overlap_pairs[near_overlap_pairs["analysis_celltype"] == cell_type]
        edges = list(local_pairs[["signature_a", "signature_b"]].itertuples(index=False, name=None))
    clusters = connected_component_table(
        group["signature"], edges, prefix=f"gene_overlap_{canonical_token(cell_type)}"
    ).rename(columns={"member": "signature"})
    clusters.insert(0, "analysis_celltype", cell_type)
    overlap_cluster_parts.append(clusters)
gene_overlap_clusters = pd.concat(overlap_cluster_parts, ignore_index=True)

catalog_export = target_catalog.drop(columns=["positive_genes", "negative_genes", "all_genes"]).copy()
catalog_export["positive_genes"] = target_catalog["positive_genes"].map(lambda genes: ";".join(genes))
catalog_export["negative_genes"] = target_catalog["negative_genes"].map(lambda genes: ";".join(genes))
catalog_export["all_genes"] = target_catalog["all_genes"].map(lambda genes: ";".join(genes))
atomic_to_csv(matrix_targets, AUDIT_DIR / "matrix_targets.csv")
atomic_to_csv(catalog_export, AUDIT_DIR / "target_signature_catalog.csv")
atomic_to_csv(exact_duplicate_drops.drop(columns=["positive_genes", "negative_genes", "all_genes"]), AUDIT_DIR / "exact_signature_duplicates_removed_target.csv")
atomic_to_csv(near_overlap_pairs, AUDIT_DIR / "near_overlap_pairs_target.csv")
atomic_to_csv(gene_overlap_clusters, AUDIT_DIR / "gene_overlap_clusters_target.csv")

signature_audit = pd.DataFrame([{
    "target_catalog_rows_before_exact_dedup": len(target_catalog_pre),
    "exact_signed_gene_set_duplicates_removed": len(exact_duplicate_drops),
    "retained_signatures": len(target_catalog),
    "near_overlap_pairs_gt_0_50": len(near_overlap_pairs),
    "non_singleton_gene_overlap_clusters": int(
        gene_overlap_clusters.loc[~gene_overlap_clusters["is_singleton"], "cluster_id"].nunique()
    ),
    "signatures_in_non_singleton_gene_overlap_clusters": int((~gene_overlap_clusters["is_singleton"]).sum()),
    "analysis_celltypes": matrix_targets["analysis_celltype"].nunique(),
    "matrix_files": matrix_targets["matrix_path"].nunique(),
}])
atomic_to_csv(signature_audit, AUDIT_DIR / "signature_audit_summary.csv")
display(signature_audit)
display(matrix_targets)

In [ ]:




AUDIT_VERSION = "bayesprism-initial-issue-verification-v1.3"

# The mapping CSV uses two distinct identities:
#   analysis_celltype = signature-catalogue key (for example CD8_T_GENERAL)
#   expression_label  = BayesPrism expression component (for example CD8+_T_cells)
# Theta and Z totals must be matched with expression_label, never analysis_celltype.
matrix_targets = matrix_targets.copy()


def expression_label_from_target(row) -> str:
    existing = str(row.get("expression_label", "")).strip()
    if existing and existing.lower() != "nan":
        return existing

    source_path = str(row.get("mapping_source_path", "")).strip()
    if source_path and source_path.lower() != "nan":
        return re.sub(
            r"_expression\.csv$", "", Path(source_path).name, flags=re.IGNORECASE
        )

    matrix_path = str(row.get("matrix_path", "")).strip()
    return re.sub(
        r"_CLEAN\.(csv|parquet)$", "", Path(matrix_path).name, flags=re.IGNORECASE
    )


matrix_targets["expression_label"] = matrix_targets.apply(
    expression_label_from_target, axis=1
)
matrix_targets["expression_key"] = matrix_targets["expression_label"].map(
    canonical_token
)
matrix_targets = matrix_targets.drop_duplicates(
    ["analysis_celltype", "expression_label", "matrix_path"]
).reset_index(drop=True)

if matrix_targets["expression_label"].eq("").any():
    raise ValueError("At least one matrix lacks an original expression-component label")

target_identity_counts = matrix_targets.groupby("analysis_celltype").agg(
    n_expression_labels=("expression_label", "nunique"),
    n_matrix_paths=("matrix_path", "nunique"),
)
target_identity_conflicts = target_identity_counts[
    (target_identity_counts["n_expression_labels"] != 1)
    | (target_identity_counts["n_matrix_paths"] != 1)
]
if not target_identity_conflicts.empty:
    atomic_to_csv(
        target_identity_conflicts.reset_index(),
        AUDIT_DIR / "mapping_target_conflicts.csv",
    )
    raise ValueError(
        "A signature-catalogue cell type maps to multiple expression components or matrices; "
        "review mapping_target_conflicts.csv"
    )

# The pasted mapping generator used Corrected_Signatures_02112025.json, whereas
# this audit scores Final_Refined_Signatures_Gemini.json. Verify every mapped
# catalogue key against the final catalogue before using it.
catalog_compatibility_rows = []
for analysis_celltype in sorted(matrix_targets["analysis_celltype"].unique()):
    exact = full_catalog[
        full_catalog["catalog_cell_type"] == analysis_celltype
    ]
    canonical_candidates = full_catalog[
        full_catalog["cell_key"] == canonical_token(analysis_celltype)
    ]
    candidate_labels = sorted(
        canonical_candidates["catalog_cell_type"].unique()
    )

    if not exact.empty:
        method = "exact"
        resolved_label = analysis_celltype
        n_final_signatures = int(len(exact))
    elif len(candidate_labels) == 1:
        method = "canonical_unique_requires_review"
        resolved_label = candidate_labels[0]
        n_final_signatures = int(len(canonical_candidates))
    elif len(candidate_labels) == 0:
        method = "missing"
        resolved_label = None
        n_final_signatures = 0
    else:
        method = "ambiguous_canonical"
        resolved_label = None
        n_final_signatures = 0

    catalog_compatibility_rows.append({
        "mapping_cell_type": analysis_celltype,
        "final_catalog_cell_type": resolved_label,
        "compatibility_method": method,
        "canonical_candidates": ";".join(candidate_labels),
        "canonical_collision": len(candidate_labels) > 1,
        "n_final_signatures": n_final_signatures,
    })

mapping_catalog_compatibility = pd.DataFrame(catalog_compatibility_rows)
atomic_to_csv(
    mapping_catalog_compatibility,
    AUDIT_DIR / "mapping_vs_final_catalog_compatibility.csv",
)
display(mapping_catalog_compatibility)

unresolved_catalog = mapping_catalog_compatibility[
    mapping_catalog_compatibility["compatibility_method"].isin(
        ["missing", "ambiguous_canonical"]
    )
]
if not unresolved_catalog.empty:
    raise ValueError(
        "At least one mapping key is absent or ambiguous in "
        "Final_Refined_Signatures_Gemini.json; review "
        "mapping_vs_final_catalog_compatibility.csv"
    )
if (
    mapping_catalog_compatibility["compatibility_method"]
    == "canonical_unique_requires_review"
).any():
    warnings.warn(
        "At least one mapping key matched the final catalogue only after "
        "canonicalization. Review mapping_vs_final_catalog_compatibility.csv."
    )

# Strict expression-component -> theta/Z-total mapping.
theta_key_to_columns = {}
for column in theta_target.columns:
    theta_key_to_columns.setdefault(canonical_token(column), []).append(column)

z_key_to_columns = {}
for column in z_totals_target.columns:
    z_key_to_columns.setdefault(canonical_token(column), []).append(column)

theta_overrides = globals().get("THETA_NAME_OVERRIDES", {})
z_overrides = globals().get("Z_NAME_OVERRIDES", {})
theta_map_rows = []

for target in matrix_targets.sort_values(
    ["analysis_celltype", "expression_label", "matrix_path"]
).itertuples(index=False):
    analysis_celltype = target.analysis_celltype
    analysis_key = canonical_token(analysis_celltype)
    expression_label = target.expression_label
    expression_key = canonical_token(expression_label)

    theta_override = theta_overrides.get(
        expression_label, theta_overrides.get(expression_key)
    )
    theta_column = None
    if theta_override is not None:
        if theta_override not in theta_target.columns:
            raise KeyError(
                f"Theta override for {expression_label} is not an exact theta "
                f"column: {theta_override}"
            )
        theta_column = theta_override
        theta_method = "reviewed_override"
    elif len(theta_key_to_columns.get(expression_key, [])) == 1:
        theta_column = theta_key_to_columns[expression_key][0]
        theta_method = "canonical_exact"
    elif len(theta_key_to_columns.get(expression_key, [])) > 1:
        theta_method = "ambiguous_canonical"
    else:
        theta_method = "unmatched"

    theta_suggestion_keys = get_close_matches(
        expression_key, list(theta_key_to_columns), n=3, cutoff=0.45
    )

    z_override = z_overrides.get(
        expression_label, z_overrides.get(expression_key)
    )
    z_total_column = None
    z_lookup_key = (
        canonical_token(theta_column)
        if theta_column is not None
        else expression_key
    )
    if z_override is not None:
        if z_override not in z_totals_target.columns:
            raise KeyError(
                f"Z override for {expression_label} is not an exact Z-total "
                f"column: {z_override}"
            )
        z_total_column = z_override
        z_method = "reviewed_override"
    elif len(z_key_to_columns.get(z_lookup_key, [])) == 1:
        z_total_column = z_key_to_columns[z_lookup_key][0]
        z_method = "canonical_exact"
    elif len(z_key_to_columns.get(z_lookup_key, [])) > 1:
        z_method = "ambiguous_canonical"
    else:
        z_method = "unmatched"

    z_suggestion_keys = get_close_matches(
        z_lookup_key, list(z_key_to_columns), n=3, cutoff=0.45
    )
    theta_map_rows.append({
        "analysis_celltype": analysis_celltype,
        "analysis_key": analysis_key,
        "expression_label": expression_label,
        "expression_key": expression_key,
        "theta_column": theta_column,
        "theta_method": theta_method,
        "suggested_theta_columns": ";".join(
            theta_key_to_columns[key][0] for key in theta_suggestion_keys
        ),
        "z_total_column": z_total_column,
        "z_method": z_method,
        "suggested_z_columns": ";".join(
            z_key_to_columns[key][0] for key in z_suggestion_keys
        ),
    })

theta_name_map = pd.DataFrame(theta_map_rows)
atomic_to_csv(theta_name_map, AUDIT_DIR / "theta_name_mapping.csv")
display(theta_name_map)

if theta_name_map[["theta_column", "z_total_column"]].isna().any().any():
    raise ValueError(
        "At least one original expression component lacks an exact theta or "
        "Z-total match. Review suggestions; add only reviewed overrides keyed "
        "by expression_label; then rerun this cell."
    )

resolved_counts = theta_name_map.groupby("analysis_celltype").agg(
    n_expression_labels=("expression_label", "nunique"),
    n_theta_columns=("theta_column", "nunique"),
    n_z_total_columns=("z_total_column", "nunique"),
)
resolved_conflicts = resolved_counts[
    (resolved_counts["n_expression_labels"] != 1)
    | (resolved_counts["n_theta_columns"] != 1)
    | (resolved_counts["n_z_total_columns"] != 1)
]
if not resolved_conflicts.empty:
    atomic_to_csv(
        resolved_conflicts.reset_index(),
        AUDIT_DIR / "theta_mapping_conflicts.csv",
    )
    raise ValueError(
        "A signature-catalogue cell type resolves to multiple expression/theta/Z "
        "components; review theta_mapping_conflicts.csv"
    )

theta_unique = theta_name_map.drop_duplicates("analysis_celltype")
THETA_BY_CELLTYPE = theta_unique.set_index("analysis_celltype")[
    "theta_column"
].to_dict()
Z_TOTAL_BY_CELLTYPE = theta_unique.set_index("analysis_celltype")[
    "z_total_column"
].to_dict()

print("PASS: signature identities were kept separate from expression/theta identities")


In [ ]:
# 10. Audit matrices and build score variants without fitting a Bayesian model
def read_clean_matrix_preserve_missingness(path: Path):
    if path.suffix.lower() in {".parquet", ".parq", ".pq"}:
        frame = pd.read_parquet(path)
    else:
        frame = pd.read_csv(path, low_memory=False)
    if frame.shape[1] < 2:
        raise ValueError(f"Expression matrix has fewer than two columns: {path}")
    id_column = frame.columns[0]
    ids = frame[id_column].map(to_base_id)
    if ids.isna().any():
        raise ValueError(f"Missing patient IDs in {path}")
    values = frame.drop(columns=[id_column]).copy()
    values.columns = [str(column).upper() for column in values.columns]
    values = values.apply(pd.to_numeric, errors="coerce")
    values = values.T.groupby(level=0, sort=False).median().T
    values.index = ids
    values = values.groupby(level=0, sort=True).median()
    return values


def impute_and_gene_z(values: pd.DataFrame):
    missing_fraction = values.isna().mean(axis=0)
    retained = missing_fraction[missing_fraction <= MAX_GENE_MISSING_FRACTION].index
    work = values.loc[:, retained].copy()
    work = work.loc[:, ~work.isna().all(axis=0)]
    work = work.fillna(work.median(axis=0))
    means = work.mean(axis=0)
    standard_deviations = work.std(axis=0, ddof=1)
    usable = standard_deviations[np.isfinite(standard_deviations) & (standard_deviations > 0)].index
    z = (work[usable] - means[usable]) / standard_deviations[usable]
    return z, missing_fraction


def score_signatures(
    gene_z: pd.DataFrame,
    source_values: pd.DataFrame,
    records: pd.DataFrame,
    cell_type: str,
    variant: str,
    matrix_path: Path,
):
    available = set(gene_z.columns)
    score_parts, metadata_rows = [], []
    for row in records.itertuples(index=False):
        positive = [gene for gene in row.positive_genes if gene in available]
        negative = [gene for gene in row.negative_genes if gene in available]
        genes_used = sorted(set(positive) | set(negative))
        n_available = len(genes_used)
        coverage = n_available / row.n_genes if row.n_genes else 0.0
        included = n_available >= MIN_SIGNATURE_GENES and coverage >= MIN_GENE_COVERAGE
        feature = f"{cell_type}||{row.signature}"
        signature_missing = source_values.reindex(index=gene_z.index, columns=genes_used).isna()
        n_signature_values = int(signature_missing.size)
        n_imputed_values = int(signature_missing.sum().sum())
        metadata_rows.append({
            "CellType": cell_type,
            "Signature": row.signature,
            "feature": feature,
            "variant": variant,
            "n_genes_total": row.n_genes,
            "n_genes_available": n_available,
            "coverage": coverage,
            "included": included,
            "n_signature_values": n_signature_values,
            "n_imputed_values": n_imputed_values,
            "fraction_imputed_values": n_imputed_values / max(1, n_signature_values),
            "patients_with_any_imputation": int(signature_missing.any(axis=1).sum()) if genes_used else 0,
            "max_imputed_genes_in_one_patient": int(signature_missing.sum(axis=1).max()) if genes_used else 0,
            "matrix_path": str(matrix_path),
        })
        if not included:
            continue
        positive_score = gene_z[positive].mean(axis=1) if positive else pd.Series(0.0, index=gene_z.index)
        negative_score = gene_z[negative].mean(axis=1) if negative else pd.Series(0.0, index=gene_z.index)
        score = positive_score - negative_score if negative else positive_score
        score_parts.append(pd.DataFrame({
            "base_sample_id": score.index,
            "CellType": cell_type,
            "Signature": row.signature,
            "feature": feature,
            "variant": variant,
            "score": score.to_numpy(),
        }))
    return score_parts, metadata_rows


cohort_index = pd.Index(cohort["base_sample_id"], name="base_sample_id")
theta_aligned = theta_target.reindex(cohort_index)
theta_cv_aligned = theta_cv_target.reindex(cohort_index) if not theta_cv_target.empty else pd.DataFrame(index=cohort_index)
score_parts_all, score_metadata_rows = [], []
matrix_summary_rows, matrix_provenance_rows = [], []
gene_qc_parts, patient_qc_parts, clipping_rows = [], [], []

for target in matrix_targets.itertuples(index=False):
    cell_type = target.analysis_celltype
    matrix_path = Path(target.matrix_path)
    theta_column = THETA_BY_CELLTYPE[cell_type]
    z_total_column = Z_TOTAL_BY_CELLTYPE[cell_type]
    values_original = read_clean_matrix_preserve_missingness(matrix_path)
    values = values_original.reindex(cohort_index)

    negative_count = int((values < -1e-10).sum().sum())
    if negative_count:
        raise ValueError(f"Negative BayesPrism allocated values found in {matrix_path}")

    row_totals = values.sum(axis=1, min_count=1)
    rds_component_totals = z_totals_target[z_total_column].reindex(cohort_index)
    total_pairs = pd.DataFrame({
        "clean_total": row_totals,
        "rds_total": rds_component_totals,
    }).replace([np.inf, -np.inf], np.nan).dropna()
    total_difference = total_pairs["clean_total"] - total_pairs["rds_total"]
    total_relative_error = total_difference.abs() / total_pairs["rds_total"].abs().clip(lower=1e-12)
    total_rho, total_p, total_n = safe_spearman(
        total_pairs["clean_total"], total_pairs["rds_total"]
    )
    clean_gene_set = set(values_original.columns)
    genes_missing_from_clean = sorted(rds_z_gene_set - clean_gene_set)
    genes_extra_in_clean = sorted(clean_gene_set - rds_z_gene_set)
    matrix_provenance_rows.append({
        "CellType": cell_type,
        "matrix_path": str(matrix_path),
        "theta_column": theta_column,
        "z_total_column": z_total_column,
        "n_clean_patients": int(values_original.shape[0]),
        "n_rds_z_patients": int(z_totals_target[z_total_column].notna().sum()),
        "n_total_pairs": int(total_n),
        "n_clean_genes": int(len(clean_gene_set)),
        "n_rds_z_genes": int(len(rds_z_gene_set)),
        "genes_missing_from_clean": int(len(genes_missing_from_clean)),
        "genes_extra_in_clean": int(len(genes_extra_in_clean)),
        "missing_gene_examples": ";".join(genes_missing_from_clean[:20]),
        "extra_gene_examples": ";".join(genes_extra_in_clean[:20]),
        "total_spearman_rho": total_rho,
        "total_spearman_p_value": total_p,
        "max_abs_total_difference": float(total_difference.abs().max()) if len(total_pairs) else np.nan,
        "max_relative_total_error": float(total_relative_error.max()) if len(total_pairs) else np.nan,
        "median_clean_to_rds_total_ratio": float(
            (total_pairs["clean_total"] / total_pairs["rds_total"].replace(0, np.nan)).median()
        ) if len(total_pairs) else np.nan,
        "totals_allclose": bool(
            len(total_pairs) == EXPECTED_PATIENTS
            and np.allclose(
                total_pairs["clean_total"], total_pairs["rds_total"],
                rtol=MAX_RDS_TOTAL_RELATIVE_ERROR, atol=1e-8,
            )
        ),
    })
    raw_z, raw_missing_fraction = impute_and_gene_z(values)
    within = values.div(row_totals.replace(0, np.nan), axis=0) * 1_000_000.0
    within = np.log1p(within)
    within_z, within_missing_fraction = impute_and_gene_z(within)

    missing = values.isna()
    zero = values.eq(0) & values.notna()
    n_values = int(values.size)
    matrix_summary_rows.append({
        "CellType": cell_type,
        "matrix_path": str(matrix_path),
        "n_patients_original": int(values_original.shape[0]),
        "n_patients_in_cohort": int(values.notna().any(axis=1).sum()),
        "n_genes": int(values.shape[1]),
        "negative_values": negative_count,
        "missing_values": int(missing.sum().sum()),
        "missing_fraction": float(missing.sum().sum() / n_values),
        "zero_values": int(zero.sum().sum()),
        "zero_fraction_nonmissing": float(zero.sum().sum() / max(1, values.notna().sum().sum())),
        "zero_total_patients": int((row_totals == 0).sum()),
        "median_component_total": float(row_totals.median()),
        "theta_column": theta_column,
        "z_total_column": z_total_column,
        "median_theta": float(theta_aligned[theta_column].median()),
        "fraction_theta_below_threshold": float((theta_aligned[theta_column] < LOW_THETA_THRESHOLD).mean()),
        "median_theta_cv": float(theta_cv_aligned[theta_column].median()) if theta_column in theta_cv_aligned else np.nan,
        "fraction_theta_cv_above_threshold": float((theta_cv_aligned[theta_column] > HIGH_THETA_CV_THRESHOLD).mean()) if theta_column in theta_cv_aligned else np.nan,
    })

    gene_qc_parts.append(pd.DataFrame({
        "CellType": cell_type,
        "gene": values.columns,
        "missing_fraction": values.isna().mean(axis=0).to_numpy(),
        "zero_fraction_nonmissing": [
            float((values[column] == 0).sum() / max(1, values[column].notna().sum()))
            for column in values.columns
        ],
        "mean_allocated": values.mean(axis=0).to_numpy(),
        "sd_allocated": values.std(axis=0, ddof=1).to_numpy(),
        "removed_by_missingness_rule": (raw_missing_fraction > MAX_GENE_MISSING_FRACTION).reindex(values.columns, fill_value=True).to_numpy(),
    }))

    patient_qc = pd.DataFrame({
        "base_sample_id": cohort_index,
        "CellType": cell_type,
        "missing_fraction": values.isna().mean(axis=1).to_numpy(),
        "zero_fraction_nonmissing": [
            float((values.loc[index] == 0).sum() / max(1, values.loc[index].notna().sum()))
            for index in values.index
        ],
        "component_total": row_totals.to_numpy(),
        "rds_component_total": rds_component_totals.to_numpy(),
        "component_total_relative_error": (
            (row_totals - rds_component_totals).abs()
            / rds_component_totals.abs().clip(lower=1e-12)
        ).to_numpy(),
        "theta": theta_aligned[theta_column].to_numpy(),
        "theta_cv": theta_cv_aligned[theta_column].to_numpy() if theta_column in theta_cv_aligned else np.nan,
    })
    patient_qc_parts.append(patient_qc)

    records = target_catalog[target_catalog["analysis_celltype"] == cell_type]
    clipping_strata = cohort.set_index("base_sample_id")[["bmi_group"]].join(
        theta_aligned[[theta_column]].rename(columns={theta_column: "theta"}), how="left"
    )
    theta_quartile_number = pd.qcut(
        clipping_strata["theta"], q=4, labels=False, duplicates="drop"
    )
    clipping_strata["theta_quartile"] = theta_quartile_number.map(
        lambda value: f"Q{int(value) + 1}" if pd.notna(value) else np.nan
    )
    for endpoint_name, z_unclipped, source_values in (
        ("raw", raw_z, values),
        ("within_component_logcpm", within_z, within),
    ):
        for cap in CLIP_CAPS:
            variant = f"{endpoint_name}_z_cap{int(cap)}"
            stratum_groups = [("all", "all", clipping_strata.index)]
            stratum_groups.extend(
                ("bmi_group", str(group_name), group.index)
                for group_name, group in clipping_strata.groupby("bmi_group", observed=True)
            )
            stratum_groups.extend(
                ("theta_quartile", str(group_name), group.index)
                for group_name, group in clipping_strata.dropna(subset=["theta_quartile"]).groupby(
                    "theta_quartile", observed=True
                )
            )
            for stratum_type, stratum, group_ids in stratum_groups:
                group_mask = z_unclipped.index.isin(group_ids)
                group_values = z_unclipped.loc[group_mask].to_numpy(dtype=float)
                clipping_rows.append({
                    "CellType": cell_type,
                    "endpoint": endpoint_name,
                    "cap": cap,
                    "stratum_type": stratum_type,
                    "stratum": stratum,
                    "n_patients": int(group_mask.sum()),
                    "n_finite_gene_values": int(np.isfinite(group_values).sum()),
                    "n_values_beyond_cap": int((np.abs(group_values) > cap).sum()),
                    "fraction_values_beyond_cap": float((np.abs(group_values) > cap).sum() / max(1, np.isfinite(group_values).sum())),
                })
            z_capped = z_unclipped.clip(-cap, cap)
            parts, rows = score_signatures(
                z_capped, source_values, records, cell_type, variant, matrix_path
            )
            score_parts_all.extend(parts)
            score_metadata_rows.extend(rows)

    del values_original, values, raw_z, within, within_z
    gc.collect()
    print("Audited", cell_type, "from", matrix_path.name)

matrix_summary = pd.DataFrame(matrix_summary_rows)
matrix_rds_provenance = pd.DataFrame(matrix_provenance_rows)
gene_qc = pd.concat(gene_qc_parts, ignore_index=True)
patient_qc = pd.concat(patient_qc_parts, ignore_index=True)
clipping_audit = pd.DataFrame(clipping_rows)
score_metadata = pd.DataFrame(score_metadata_rows)
audit_scores = pd.concat(score_parts_all, ignore_index=True)

duplicate_key = ["base_sample_id", "feature", "variant"]
if audit_scores.duplicated(duplicate_key).any():
    spread = audit_scores.groupby(duplicate_key)["score"].agg(lambda x: float(x.max() - x.min()))
    if (spread > 1e-10).any():
        raise ValueError("Conflicting duplicate patient-feature-variant scores")
    audit_scores = audit_scores.drop_duplicates(duplicate_key)

atomic_to_csv(matrix_summary, AUDIT_DIR / "matrix_qc_summary.csv")
atomic_to_csv(matrix_rds_provenance, AUDIT_DIR / "clean_matrix_vs_rds_z.csv")
atomic_to_csv(gene_qc, AUDIT_DIR / "gene_qc.csv.gz", compression="gzip")
atomic_to_csv(patient_qc, AUDIT_DIR / "patient_qc.csv")
atomic_to_csv(clipping_audit, AUDIT_DIR / "clipping_audit.csv")
atomic_to_csv(score_metadata, AUDIT_DIR / "score_metadata_all_variants.csv")
if SAVE_FULL_SCORE_TABLE:
    atomic_to_csv(audit_scores, AUDIT_DIR / "audit_scores_all_variants.csv.gz", compression="gzip")

display(matrix_summary)
display(matrix_rds_provenance)
print("Scores:", len(audit_scores), "rows;", audit_scores["feature"].nunique(), "features")

In [ ]:
# 11. OLS helpers and theta-BMI associations
def ols_coefficient(frame: pd.DataFrame, outcome: str, exposure: str, covariates=("age_z", "sex_centered")):
    columns = [outcome, exposure, *covariates]
    work = frame[columns].replace([np.inf, -np.inf], np.nan).dropna()
    if len(work) <= len(columns) + 1 or work[outcome].nunique() < 2 or work[exposure].nunique() < 2:
        return {"n": len(work), "beta": np.nan, "se": np.nan, "p_value": np.nan}
    y = work[outcome].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(work)), *[work[column].to_numpy(dtype=float) for column in [exposure, *covariates]]])
    beta = np.linalg.pinv(X) @ y
    residual = y - X @ beta
    rank = np.linalg.matrix_rank(X)
    dof = len(y) - rank
    if dof <= 0:
        return {"n": len(work), "beta": float(beta[1]), "se": np.nan, "p_value": np.nan}
    sigma2 = float(residual @ residual / dof)
    covariance = sigma2 * np.linalg.pinv(X.T @ X)
    se = float(np.sqrt(max(covariance[1, 1], 0)))
    t_value = float(beta[1] / se) if se > 0 else np.nan
    p_value = float(2 * stats.t.sf(abs(t_value), dof)) if np.isfinite(t_value) else np.nan
    return {"n": len(work), "beta": float(beta[1]), "se": se, "p_value": p_value}


theta_join = cohort.set_index("base_sample_id").join(theta_target, how="inner")
theta_bmi_rows = []
for component in theta_target.columns:
    work = theta_join[["bmi", "bmi_z", "age_z", "sex_centered", component]].copy()
    work["theta_asin_sqrt"] = np.arcsin(np.sqrt(work[component].clip(0, 1)))
    fit = ols_coefficient(work, "theta_asin_sqrt", "bmi_z")
    rho, spearman_p, n_spearman = safe_spearman(work[component], work["bmi"])
    theta_bmi_rows.append({
        "component": component,
        **fit,
        "spearman_rho": rho,
        "spearman_p_value": spearman_p,
        "spearman_n": n_spearman,
        "mean_theta": float(work[component].mean()),
        "median_theta": float(work[component].median()),
        "fraction_below_low_theta": float((work[component] < LOW_THETA_THRESHOLD).mean()),
        "median_theta_cv": float(theta_cv_target[component].median()) if component in theta_cv_target else np.nan,
    })
theta_bmi_associations = pd.DataFrame(theta_bmi_rows)
theta_bmi_associations["q_value"] = bh_adjust(theta_bmi_associations["p_value"])
atomic_to_csv(theta_bmi_associations.sort_values("q_value"), AUDIT_DIR / "theta_bmi_associations.csv")
display(theta_bmi_associations.sort_values("q_value").head(20))

In [ ]:
# 12. Score slopes, score-theta associations, missingness associations, and variant concordance
scores_with_cohort = audit_scores.merge(cohort, on="base_sample_id", how="inner", validate="many_to_one")
score_slope_rows = []
score_theta_rows = []
concordance_rows = []

def residualize(y, design):
    work = pd.concat([pd.Series(y, name="y"), design], axis=1).dropna()
    if len(work) < design.shape[1] + 3:
        return pd.Series(dtype=float)
    X = np.column_stack([np.ones(len(work)), work[design.columns].to_numpy(dtype=float)])
    values = work["y"].to_numpy(dtype=float)
    residual = values - X @ (np.linalg.pinv(X) @ values)
    return pd.Series(residual, index=work.index)


for (feature, variant), group in scores_with_cohort.groupby(["feature", "variant"], sort=False):
    group = group.copy()
    score_sd = group["score"].std(ddof=1)
    group["score_z"] = (group["score"] - group["score"].mean()) / score_sd if score_sd > 0 else np.nan
    fit = ols_coefficient(group, "score_z", "bmi_z")
    theta_column = THETA_BY_CELLTYPE[group["CellType"].iloc[0]]
    theta_series = theta_target[theta_column].rename("theta")
    group["theta"] = group["base_sample_id"].map(theta_series)
    fit_theta_adjusted = ols_coefficient(
        group,
        "score_z",
        "bmi_z",
        covariates=("age_z", "sex_centered", "theta"),
    )
    score_slope_rows.append({
        "feature": feature,
        "CellType": group["CellType"].iloc[0],
        "Signature": group["Signature"].iloc[0],
        "variant": variant,
        **fit,
        "theta_adjusted_beta": fit_theta_adjusted["beta"],
        "theta_adjusted_se": fit_theta_adjusted["se"],
        "theta_adjusted_p_value": fit_theta_adjusted["p_value"],
        "theta_adjustment_beta_delta": fit_theta_adjusted["beta"] - fit["beta"],
    })

    local = group.set_index("base_sample_id")
    rho, pvalue, n = safe_spearman(local["score"], local["theta"])
    design = local[["bmi_z", "age_z", "sex_centered"]]
    score_resid = residualize(local["score"], design)
    theta_resid = residualize(local["theta"], design)
    common = score_resid.index.intersection(theta_resid.index)
    partial_r = float(np.corrcoef(score_resid.loc[common], theta_resid.loc[common])[0, 1]) if len(common) >= 4 else np.nan
    score_theta_rows.append({
        "feature": feature,
        "CellType": group["CellType"].iloc[0],
        "Signature": group["Signature"].iloc[0],
        "variant": variant,
        "theta_column": theta_column,
        "n": n,
        "spearman_rho": rho,
        "p_value": pvalue,
        "partial_r_after_bmi_age_sex": partial_r,
    })

score_slopes = pd.DataFrame(score_slope_rows)
score_slopes["q_value"] = score_slopes.groupby("variant")["p_value"].transform(bh_adjust)
score_slopes["theta_adjusted_q_value"] = score_slopes.groupby("variant")[
    "theta_adjusted_p_value"
].transform(bh_adjust)
score_theta_associations = pd.DataFrame(score_theta_rows)
score_theta_associations["q_value"] = score_theta_associations.groupby("variant")["p_value"].transform(bh_adjust)

for feature, group in audit_scores.groupby("feature", sort=False):
    pivot = group.pivot(index="base_sample_id", columns="variant", values="score")
    def corr(left, right):
        if left not in pivot or right not in pivot:
            return np.nan
        return float(pivot[left].corr(pivot[right]))
    concordance_rows.append({
        "feature": feature,
        "CellType": group["CellType"].iloc[0],
        "Signature": group["Signature"].iloc[0],
        "raw_cap3_vs_cap5_r": corr("raw_z_cap3", "raw_z_cap5"),
        "within_cap3_vs_cap5_r": corr("within_component_logcpm_z_cap3", "within_component_logcpm_z_cap5"),
        "raw_vs_within_cap3_r": corr("raw_z_cap3", "within_component_logcpm_z_cap3"),
        "raw_vs_within_cap5_r": corr("raw_z_cap5", "within_component_logcpm_z_cap5"),
    })
score_variant_concordance = pd.DataFrame(concordance_rows)

slope_wide = score_slopes.pivot(index="feature", columns="variant", values="beta")
if {"raw_z_cap3", "raw_z_cap5"}.issubset(slope_wide.columns):
    slope_delta = (slope_wide["raw_z_cap5"] - slope_wide["raw_z_cap3"]).rename("raw_slope_delta_cap5_minus_cap3")
    score_variant_concordance = score_variant_concordance.merge(slope_delta, on="feature", how="left")
if {"within_component_logcpm_z_cap3", "raw_z_cap3"}.issubset(slope_wide.columns):
    normalization_delta = (slope_wide["within_component_logcpm_z_cap3"] - slope_wide["raw_z_cap3"]).rename("slope_delta_within_minus_raw_cap3")
    score_variant_concordance = score_variant_concordance.merge(normalization_delta, on="feature", how="left")

missingness_rows = []
patient_qc_joined = patient_qc.merge(cohort, on="base_sample_id", how="left", validate="many_to_one")
for cell_type, group in patient_qc_joined.groupby("CellType"):
    rho_bmi, p_bmi, n_bmi = safe_spearman(group["missing_fraction"], group["bmi"])
    rho_theta, p_theta, n_theta = safe_spearman(group["missing_fraction"], group["theta"])
    zero_rho_bmi, zero_p_bmi, zero_n_bmi = safe_spearman(group["zero_fraction_nonmissing"], group["bmi"])
    zero_rho_theta, zero_p_theta, zero_n_theta = safe_spearman(group["zero_fraction_nonmissing"], group["theta"])
    total_rho_theta, total_p_theta, total_n_theta = safe_spearman(group["component_total"], group["theta"])
    adjusted_missing = ols_coefficient(group, "missing_fraction", "bmi_z")
    adjusted_zero = ols_coefficient(group, "zero_fraction_nonmissing", "bmi_z")
    missingness_rows.append({
        "CellType": cell_type,
        "missing_vs_bmi_rho": rho_bmi,
        "missing_vs_bmi_p": p_bmi,
        "missing_vs_bmi_n": n_bmi,
        "missing_vs_theta_rho": rho_theta,
        "missing_vs_theta_p": p_theta,
        "missing_vs_theta_n": n_theta,
        "adjusted_missing_vs_bmi_beta": adjusted_missing["beta"],
        "adjusted_missing_vs_bmi_p": adjusted_missing["p_value"],
        "zero_vs_bmi_rho": zero_rho_bmi,
        "zero_vs_bmi_p": zero_p_bmi,
        "zero_vs_bmi_n": zero_n_bmi,
        "zero_vs_theta_rho": zero_rho_theta,
        "zero_vs_theta_p": zero_p_theta,
        "zero_vs_theta_n": zero_n_theta,
        "adjusted_zero_vs_bmi_beta": adjusted_zero["beta"],
        "adjusted_zero_vs_bmi_p": adjusted_zero["p_value"],
        "component_total_vs_theta_rho": total_rho_theta,
        "component_total_vs_theta_p": total_p_theta,
        "component_total_vs_theta_n": total_n_theta,
    })
missingness_associations = pd.DataFrame(missingness_rows)
for p_column in (
    "missing_vs_bmi_p", "missing_vs_theta_p", "adjusted_missing_vs_bmi_p",
    "zero_vs_bmi_p", "zero_vs_theta_p", "adjusted_zero_vs_bmi_p",
):
    missingness_associations[p_column.replace("_p", "_q")] = bh_adjust(
        missingness_associations[p_column]
    )

atomic_to_csv(score_slopes, AUDIT_DIR / "descriptive_score_bmi_slopes.csv")
atomic_to_csv(score_theta_associations, AUDIT_DIR / "score_theta_associations.csv")
atomic_to_csv(score_variant_concordance, AUDIT_DIR / "score_variant_concordance.csv")
atomic_to_csv(missingness_associations, AUDIT_DIR / "missingness_associations.csv")
display(score_variant_concordance.describe(include="all").T)

In [ ]:
# 13. Within-cell-type score correlations before and after theta/covariate projection
raw3 = audit_scores[audit_scores["variant"] == "raw_z_cap3"].copy()
empirical_rows = []
overlap_lookup = set()
if not near_overlap_pairs.empty:
    for row in near_overlap_pairs.itertuples(index=False):
        overlap_lookup.add((row.analysis_celltype, *sorted([row.signature_a, row.signature_b])))

cohort_design = cohort.set_index("base_sample_id")[["bmi_z", "age_z", "sex_centered"]]
for cell_type, group in raw3.groupby("CellType"):
    pivot = group.pivot(index="base_sample_id", columns="Signature", values="score").sort_index()
    if pivot.shape[1] < 2:
        continue
    raw_corr = pivot.corr()

    theta_column = THETA_BY_CELLTYPE[cell_type]
    design = cohort_design.join(theta_target[[theta_column]].rename(columns={theta_column: "theta"}), how="inner")
    common_ids = pivot.index.intersection(design.index)
    Y = pivot.loc[common_ids]
    X_frame = design.loc[common_ids]
    valid_rows = Y.notna().all(axis=1) & X_frame.notna().all(axis=1)
    Y_values = Y.loc[valid_rows].to_numpy(dtype=float)
    X_values = np.column_stack([np.ones(valid_rows.sum()), X_frame.loc[valid_rows].to_numpy(dtype=float)])
    residual_values = Y_values - X_values @ (np.linalg.pinv(X_values) @ Y_values)
    residual_corr = pd.DataFrame(residual_values, columns=Y.columns).corr()

    names = list(pivot.columns)
    for left in range(len(names)):
        for right in range(left + 1, len(names)):
            sig_a, sig_b = names[left], names[right]
            r_raw = float(raw_corr.loc[sig_a, sig_b])
            r_resid = float(residual_corr.loc[sig_a, sig_b])
            if max(abs(r_raw), abs(r_resid)) >= EMPIRICAL_CORRELATION_THRESHOLD:
                empirical_rows.append({
                    "CellType": cell_type,
                    "signature_a": sig_a,
                    "signature_b": sig_b,
                    "raw_r": r_raw,
                    "residual_r_after_bmi_age_sex_theta": r_resid,
                    "gene_overlap_gt_0_50": (cell_type, *sorted([sig_a, sig_b])) in overlap_lookup,
                })

empirical_correlation_pairs = pd.DataFrame(empirical_rows)
empirical_cluster_parts = []
for correlation_kind, correlation_column in (
    ("raw", "raw_r"),
    ("residual_after_bmi_age_sex_theta", "residual_r_after_bmi_age_sex_theta"),
):
    for cell_type, group in raw3.groupby("CellType"):
        if empirical_correlation_pairs.empty:
            edges = []
        else:
            local_pairs = empirical_correlation_pairs[
                (empirical_correlation_pairs["CellType"] == cell_type)
                & (empirical_correlation_pairs[correlation_column].abs() >= EMPIRICAL_CORRELATION_THRESHOLD)
            ]
            edges = list(local_pairs[["signature_a", "signature_b"]].itertuples(index=False, name=None))
        clusters = connected_component_table(
            group["Signature"],
            edges,
            prefix=f"{correlation_kind}_{canonical_token(cell_type)}",
        ).rename(columns={"member": "Signature"})
        clusters.insert(0, "CellType", cell_type)
        clusters.insert(1, "correlation_kind", correlation_kind)
        empirical_cluster_parts.append(clusters)
empirical_correlation_clusters = pd.concat(empirical_cluster_parts, ignore_index=True)
atomic_to_csv(empirical_correlation_pairs, AUDIT_DIR / "empirical_high_correlation_pairs.csv")
atomic_to_csv(empirical_correlation_clusters, AUDIT_DIR / "empirical_correlation_clusters.csv")
print("High-correlation pairs:", len(empirical_correlation_pairs))
if not empirical_correlation_pairs.empty:
    display(empirical_correlation_pairs.sort_values("residual_r_after_bmi_age_sex_theta", key=lambda s: s.abs(), ascending=False).head(30))

In [ ]:
# 14. Save audit figures
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

theta_plot = theta_bmi_associations.sort_values("q_value").head(min(20, len(theta_bmi_associations))).copy()
sns.barplot(data=theta_plot, x="beta", y="component", ax=axes[0, 0], color="#4472C4")
axes[0, 0].axvline(0, color="black", linewidth=0.8)
axes[0, 0].set_title("Adjusted theta–BMI slopes (top by q)")
axes[0, 0].set_xlabel("Arcsine-sqrt theta change per BMI SD")

sns.histplot(score_variant_concordance["raw_cap3_vs_cap5_r"].dropna(), bins=30, ax=axes[0, 1], color="#70AD47")
axes[0, 1].axvline(0.99, color="black", linestyle="--", linewidth=1)
axes[0, 1].set_title("Feature concordance: raw cap 3 vs cap 5")

sns.histplot(score_variant_concordance["raw_vs_within_cap3_r"].dropna(), bins=30, ax=axes[1, 0], color="#ED7D31")
axes[1, 0].set_title("Feature concordance: raw vs within-component")

reliability_plot = matrix_summary.sort_values("median_theta")
sns.scatterplot(
    data=reliability_plot,
    x="median_theta",
    y="missing_fraction",
    size="zero_fraction_nonmissing",
    hue="fraction_theta_below_threshold",
    palette="viridis",
    ax=axes[1, 1],
    legend="brief",
)
axes[1, 1].set_xscale("log")
axes[1, 1].set_title("Matrix reliability versus abundance")
axes[1, 1].set_xlabel("Median theta (log scale)")

fig.suptitle(f"BayesPrism initial issue verification — {TARGET_COMPARTMENT}", fontsize=15)
fig.tight_layout()
figure_path = PLOT_DIR / "initial_issue_verification_summary.png"
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)

In [ ]:
# 15. Generate a machine-readable and human-readable decision report
gates = []
def gate(name, status, evidence, action):
    gates.append({"gate": name, "status": status, "evidence": evidence, "action": action})

cohort_ok = len(cohort) == EXPECTED_PATIENTS and cohort_counts == EXPECTED_COHORT_COUNTS and len(theta_overlap) == EXPECTED_PATIENTS
gate(
    "patient_identity",
    "PASS" if cohort_ok else "BLOCK",
    f"cohort={len(cohort)}, theta_overlap={len(theta_overlap)}, groups={cohort_counts}",
    "Resolve IDs/cohort before any model" if not cohort_ok else "None",
)

clean_z_ok = (
    len(matrix_rds_provenance) == len(matrix_targets)
    and (matrix_rds_provenance["n_total_pairs"] == EXPECTED_PATIENTS).all()
    and (matrix_rds_provenance["genes_missing_from_clean"] == 0).all()
    and (matrix_rds_provenance["genes_extra_in_clean"] == 0).all()
    and matrix_rds_provenance["totals_allclose"].all()
)
gate(
    "clean_matrix_vs_rds_z",
    "PASS" if clean_z_ok else "BLOCK",
    (
        f"matrices={len(matrix_rds_provenance)}, "
        f"max_missing_genes={matrix_rds_provenance['genes_missing_from_clean'].max()}, "
        f"max_extra_genes={matrix_rds_provenance['genes_extra_in_clean'].max()}, "
        f"max_relative_total_error={matrix_rds_provenance['max_relative_total_error'].max():.3g}"
    ),
    "Resolve expression provenance or matrix/RDS mismatch before score rebuilding" if not clean_z_ok else "None",
)

matrix_patient_ok = (
    (matrix_summary["n_patients_original"] == EXPECTED_PATIENTS).all()
    and (matrix_summary["n_patients_in_cohort"] == EXPECTED_PATIENTS).all()
)
gate(
    "matrix_patient_coverage",
    "PASS" if matrix_patient_ok else "BLOCK",
    f"minimum original={matrix_summary['n_patients_original'].min()}, minimum cohort={matrix_summary['n_patients_in_cohort'].min()}",
    "Resolve missing/duplicate component rows before scoring" if not matrix_patient_ok else "None",
)

theta_ok = (
    theta_validation["negative_values"] == 0
    and theta_validation["nonfinite_values"] == 0
    and theta_validation["fraction_rows_within_1e_3"] == 1.0
)
gate(
    "theta_validity",
    "PASS" if theta_ok else "BLOCK",
    f"kind={CFG['theta_kind']}, components={theta_validation['n_components']}, rows_within_1e-3={theta_validation['fraction_rows_within_1e_3']:.3f}",
    "Inspect RDS orientation and slot provenance" if not theta_ok else "None",
)

z_consistency_ok = (
    not z_theta_consistency.empty
    and z_theta_consistency["spearman_rho"].median() >= 0.999
    and z_theta_consistency["max_abs_difference"].max() <= 1e-6
)
gate(
    "z_theta_structural_consistency",
    "PASS" if z_consistency_ok else "REVIEW",
    (
        f"median_rho={z_theta_consistency['spearman_rho'].median():.6f}, max_abs_diff={z_theta_consistency['max_abs_difference'].max():.3g}"
        if not z_theta_consistency.empty else "No matched Z-total/initial-theta comparison"
    ),
    "Inspect component labels, Z dimensions, and initial theta" if not z_consistency_ok else "None",
)

all_theta_names = theta_name_map[["theta_column", "z_total_column"]].notna().all().all()
gate(
    "theta_expression_label_map",
    "PASS" if all_theta_names else "BLOCK",
    (
        f"theta_matched={theta_name_map['theta_column'].notna().sum()}/{len(theta_name_map)}, "
        f"z_matched={theta_name_map['z_total_column'].notna().sum()}/{len(theta_name_map)}"
    ),
    "Add only reviewed THETA_NAME_OVERRIDES/Z_NAME_OVERRIDES" if not all_theta_names else "None",
)

gate(
    "exact_signature_duplication",
    "PASS" if len(exact_duplicate_drops) == 0 else "REVIEW",
    f"exact signed-gene-set duplicates removed in target={len(exact_duplicate_drops)}",
    "Report deterministic removals; never count them as separate discoveries" if len(exact_duplicate_drops) else "None",
)
gate(
    "near_signature_overlap",
    "PASS" if len(near_overlap_pairs) == 0 else "REVIEW",
    f"gene-overlap pairs > {NEAR_OVERLAP_THRESHOLD} in target={len(near_overlap_pairs)}",
    "Use cluster-level/representative-signature sensitivity if retained",
)

n_residual_correlated_pairs = int(
    (empirical_correlation_pairs["residual_r_after_bmi_age_sex_theta"].abs() >= EMPIRICAL_CORRELATION_THRESHOLD).sum()
) if not empirical_correlation_pairs.empty else 0
n_residual_clusters = int(
    empirical_correlation_clusters.loc[
        (empirical_correlation_clusters["correlation_kind"] == "residual_after_bmi_age_sex_theta")
        & (~empirical_correlation_clusters["is_singleton"]),
        "cluster_id",
    ].nunique()
)
gate(
    "empirical_signature_dependence",
    "PASS" if n_residual_correlated_pairs == 0 else "REVIEW",
    f"residual_abs_r_ge_{EMPIRICAL_CORRELATION_THRESHOLD} pairs={n_residual_correlated_pairs}, clusters={n_residual_clusters}",
    "Carry response-independent cluster/representative sensitivity into BFDR design" if n_residual_correlated_pairs else "None",
)

overall_missing = float(matrix_summary["missing_values"].sum() / matrix_summary["n_genes"].mul(EXPECTED_PATIENTS).sum())
max_patient_missing = float(patient_qc["missing_fraction"].max())
missing_status = "PASS" if overall_missing < 0.001 and max_patient_missing < 0.01 else "REVIEW"
gate(
    "missingness",
    missing_status,
    f"overall={overall_missing:.6f}, max_patient_component={max_patient_missing:.6f}",
    "Review BMI/theta missingness associations and exclusions" if missing_status == "REVIEW" else "None",
)

raw_imputation_metadata = score_metadata[
    (score_metadata["variant"] == "raw_z_cap3") & score_metadata["included"]
]
max_signature_imputation = float(raw_imputation_metadata["fraction_imputed_values"].max())
signature_imputation_status = "PASS" if max_signature_imputation < 0.01 else "REVIEW"
gate(
    "signature_imputation_burden",
    signature_imputation_status,
    f"maximum included-signature imputed fraction={max_signature_imputation:.6f}",
    "Use signature-level imputation exclusions/sensitivity" if signature_imputation_status == "REVIEW" else "None",
)

low_reliability = matrix_summary[
    (matrix_summary["fraction_theta_below_threshold"] > 0.20)
    | (matrix_summary["fraction_theta_cv_above_threshold"] > 0.20)
    | (matrix_summary["zero_total_patients"] > 0)
]
gate(
    "low_abundance_reliability",
    "PASS" if low_reliability.empty else "REVIEW",
    f"flagged components={len(low_reliability)}/{len(matrix_summary)}",
    "Predeclare low-theta/high-CV component exclusion sensitivity" if not low_reliability.empty else "None",
)

min_clip_corr = float(score_variant_concordance["raw_cap3_vs_cap5_r"].min())
max_clip_slope_delta = float(score_variant_concordance.get("raw_slope_delta_cap5_minus_cap3", pd.Series([np.nan])).abs().max())
clip_status = "PASS" if min_clip_corr >= 0.99 and (not np.isfinite(max_clip_slope_delta) or max_clip_slope_delta <= 0.05) else "REVIEW"
gate(
    "cap3_vs_cap5",
    clip_status,
    f"min_feature_r={min_clip_corr:.4f}, max_abs_descriptive_slope_delta={max_clip_slope_delta:.4f}",
    "Keep ±3 primary and carry ±5 sensitivity" if clip_status == "PASS" else "Predeclare both variants in model sensitivity",
)

median_raw_within = float(score_variant_concordance["raw_vs_within_cap3_r"].median())
normalization_delta = score_variant_concordance.get("slope_delta_within_minus_raw_cap3", pd.Series(dtype=float)).abs()
max_normalization_slope_delta = float(normalization_delta.max()) if len(normalization_delta) else np.nan
abundance_status = "PASS" if median_raw_within >= 0.90 and (not np.isfinite(max_normalization_slope_delta) or max_normalization_slope_delta <= 0.10) else "REVIEW"
gate(
    "abundance_weighting",
    abundance_status,
    f"median raw-vs-within r={median_raw_within:.4f}, max_abs_slope_delta={max_normalization_slope_delta:.4f}",
    "Use within-component endpoint as replacement primary or key sensitivity" if abundance_status == "REVIEW" else "Still retain theta-adjusted sensitivity",
)

n_theta_bmi = int((theta_bmi_associations["q_value"] < 0.05).sum())
gate(
    "theta_bmi_association",
    "PASS" if n_theta_bmi == 0 else "REVIEW",
    f"theta components with adjusted q<0.05={n_theta_bmi}",
    "Interpret raw-score associations as potentially compositional" if n_theta_bmi else "No strong adjusted theta signal detected; do not treat null as exact independence",
)

raw_cap3_slopes = score_slopes[score_slopes["variant"] == "raw_z_cap3"]
max_theta_adjustment_delta = float(raw_cap3_slopes["theta_adjustment_beta_delta"].abs().max())
theta_adjustment_status = "PASS" if max_theta_adjustment_delta <= 0.05 else "REVIEW"
gate(
    "score_slope_theta_adjustment",
    theta_adjustment_status,
    f"maximum raw-cap3 absolute BMI-slope change after adding theta={max_theta_adjustment_delta:.4f}",
    "Treat raw score signal as composition-sensitive and prioritize within-component endpoint" if theta_adjustment_status == "REVIEW" else "Retain theta-adjusted sensitivity",
)

next_model_requirements = pd.DataFrame([
    {
        "issue": "heterogeneous_age_sex_effects",
        "required_next_check": "hierarchical cell-type-specific age and sex coefficients",
        "stage": "replacement-model sensitivity",
        "resolved_by_this_audit": False,
    },
    {
        "issue": "heterogeneous_residual_variance",
        "required_next_check": "feature- or cell-type-specific residual scale plus posterior predictive checks",
        "stage": "replacement-model sensitivity",
        "resolved_by_this_audit": False,
    },
    {
        "issue": "patient_by_celltype_dependence",
        "required_next_check": "patient-by-cell-type random-effect sensitivity",
        "stage": "replacement-model sensitivity",
        "resolved_by_this_audit": False,
    },
    {
        "issue": "nonlinear_or_threshold_bmi",
        "required_next_check": "categorical and quadratic/spline BMI sensitivity",
        "stage": "replacement-model sensitivity",
        "resolved_by_this_audit": False,
    },
    {
        "issue": "rope_choice",
        "required_next_check": "posterior BFDR grid at ROPE 0.05, 0.10, 0.15, and 0.20",
        "stage": "post-fit decision sensitivity",
        "resolved_by_this_audit": False,
    },
    {
        "issue": "posterior_predictive_fit",
        "required_next_check": "memory-controlled posterior predictive checks by feature and cell type",
        "stage": "post-fit diagnostics",
        "resolved_by_this_audit": False,
    },
])
atomic_to_csv(next_model_requirements, AUDIT_DIR / "next_model_design_requirements.csv")

gates_df = pd.DataFrame(gates)
if (gates_df["status"] == "BLOCK").any():
    overall = "BLOCKED"
elif (gates_df["status"] == "REVIEW").any():
    overall = "REVIEW_REQUIRED"
else:
    overall = "AUDIT_CLEAR_FOR_MODEL_DESIGN"

summary = {
    "audit_version": AUDIT_VERSION,
    "mode": MODE,
    "project_repository_commit": PROJECT_REPOSITORY_COMMIT,
    "target_compartment": TARGET_COMPARTMENT,
    "module": CFG["module"],
    "level": CFG["level"],
    "theta_kind": CFG["theta_kind"],
    "rds_path": str(RDS_PATH),
    "bayesprism_version": BAYESPRISM_VERSION,
    "audit_directory": str(AUDIT_DIR),
    "overall_decision": overall,
    "n_patients": int(len(cohort)),
    "n_celltypes": int(matrix_targets["analysis_celltype"].nunique()),
    "n_features": int(audit_scores["feature"].nunique()),
    "candidate_endpoints": [
        "raw_z_cap3", "raw_z_cap5",
        "within_component_logcpm_z_cap3", "within_component_logcpm_z_cap5",
    ],
    "input_sha256": {
        row["role"]: row["sha256"]
        for row in path_audit.to_dict(orient="records")
        if isinstance(row.get("sha256"), str) and row["sha256"]
    },
    "gates": gates,
}
atomic_to_csv(gates_df, AUDIT_DIR / "decision_gates.csv")
atomic_write_text(json.dumps(summary, indent=2), AUDIT_DIR / "audit_summary.json")

report_lines = [
    f"# BayesPrism initial issue verification — {TARGET_COMPARTMENT}",
    "",
    f"- Overall decision: **{overall}**",
    f"- RDS: `{RDS_PATH}`",
    f"- Theta: `{CFG['theta_kind']}`",
    f"- Patients: {len(cohort)}",
    f"- Cell labels: {matrix_targets['analysis_celltype'].nunique()}",
    f"- Scored features: {audit_scores['feature'].nunique()}",
    "",
    "| Gate | Status | Evidence | Action |",
    "|---|---|---|---|",
]
for row in gates:
    report_lines.append(f"| {row['gate']} | {row['status']} | {row['evidence']} | {row['action']} |")
report_lines.extend([
    "",
    "This audit does not fit a Bayesian model. Resolve every BLOCK and review every REVIEW item before choosing the replacement model endpoint and sensitivity set.",
    "",
    "## Deferred replacement-model checks",
    "",
    "The following questions cannot be settled by preprocessing audit alone: heterogeneous age/sex effects, heterogeneous residual variance, patient-by-cell-type dependence, nonlinear BMI, the ROPE grid, and posterior predictive fit. Their required analyses are saved in `next_model_design_requirements.csv`.",
])
report = "\n".join(report_lines) + "\n"
atomic_write_text(report, AUDIT_DIR / "AUDIT_REPORT.md")
generated_files = [
    path for path in sorted(AUDIT_DIR.rglob("*"))
    if path.is_file() and path.name != "generated_output_manifest.csv"
]
generated_output_manifest = pd.DataFrame([
    {
        "relative_path": str(path.relative_to(AUDIT_DIR)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in generated_files
])
atomic_to_csv(generated_output_manifest, AUDIT_DIR / "generated_output_manifest.csv")
display(Markdown(report))

In [ ]:






from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


# ----------------------------- Configuration -----------------------------
AUDIT_ROOT = Path(
    "/content/drive/MyDrive/new_dysfunction/"
    "bayesprism_initial_issue_verification_v1"
)
COMPARTMENTS = ("immune_course", "immune_fine", "non_immune")



RUN_DIR_OVERRIDES = {
    "immune_course": None,
    "immune_fine": None,
    "non_immune": None,
}

NORMALIZED_VARIANTS = (
    "within_component_logcpm_z_cap3",
    "within_component_logcpm_z_cap5",
)
RAW_VARIANTS = ("raw_z_cap3", "raw_z_cap5")
CORRELATION_THRESHOLD = 0.90
LOW_THETA_THRESHOLD = 0.005
LOW_THETA_FRACTION_LIMIT = 0.20
HIGH_THETA_CV_THRESHOLD = 1.0
HIGH_THETA_CV_FRACTION_LIMIT = 0.20
ZERO_HEAVY_GENE_FRACTION = 0.20
Q_LEVEL = 0.05

REQUIRED_FILES = (
    "audit_summary.json",
    "analysis_cohort.csv",
    "matrix_qc_summary.csv",
    "patient_qc.csv",
    "theta_name_mapping.csv",
    "theta_bmi_associations.csv",
    "near_overlap_pairs_target.csv",
    "score_variant_concordance.csv",
    "descriptive_score_bmi_slopes.csv",
    "audit_scores_all_variants.csv.gz",
)


# ------------------------------- Utilities -------------------------------
def atomic_to_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    temporary.replace(path)


def atomic_write_text(text: str, path: Path) -> None:
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def read_csv_allow_empty(path: Path, columns=()) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=list(columns))


def finite_quantile(values, probability: float) -> float:
    numeric = pd.to_numeric(pd.Series(values), errors="coerce")
    numeric = numeric[np.isfinite(numeric)]
    return float(numeric.quantile(probability)) if len(numeric) else np.nan


def safe_correlation(left, right) -> float:
    pair = pd.DataFrame({"left": left, "right": right}).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if len(pair) < 4 or pair["left"].nunique() < 2 or pair["right"].nunique() < 2:
        return np.nan
    return float(pair["left"].corr(pair["right"]))


def correlation_matrix(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[0] < 4:
        return np.full((values.shape[1], values.shape[1]), np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.corrcoef(values, rowvar=False)


def residualize_matrix(values: np.ndarray, design: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    design = np.asarray(design, dtype=float)
    return values - design @ (np.linalg.pinv(design) @ values)


def connected_components(nodes, edges, prefix: str) -> pd.DataFrame:
    nodes = sorted(set(nodes))
    parent = {node: node for node in nodes}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left, root_right = find(left), find(right)
        if root_left == root_right:
            return
        keep, merge = sorted((root_left, root_right))
        parent[merge] = keep

    for left, right in edges:
        if left in parent and right in parent:
            union(left, right)

    groups = {}
    for node in nodes:
        groups.setdefault(find(node), []).append(node)

    rows = []
    ordered_groups = sorted(groups.values(), key=lambda members: (members[0], len(members)))
    for number, members in enumerate(ordered_groups, start=1):
        cluster_id = f"{prefix}_{number:04d}"
        for member in sorted(members):
            rows.append({
                "cluster_id": cluster_id,
                "feature": member,
                "cluster_size": len(members),
                "is_singleton": len(members) == 1,
            })
    return pd.DataFrame(rows)


def resolve_run_directory(compartment: str) -> Path:
    override = RUN_DIR_OVERRIDES.get(compartment)
    if override:
        candidate = Path(override)
        missing = [name for name in REQUIRED_FILES if not (candidate / name).exists()]
        if missing:
            raise FileNotFoundError(
                f"Override for {compartment} is incomplete; missing: {missing}"
            )
        return candidate

    compartment_root = AUDIT_ROOT / compartment
    if not compartment_root.exists():
        raise FileNotFoundError(f"Missing audit compartment folder: {compartment_root}")
    candidates = sorted(
        (path for path in compartment_root.iterdir() if path.is_dir()),
        key=lambda path: path.name,
        reverse=True,
    )
    for candidate in candidates:
        if all((candidate / name).exists() for name in REQUIRED_FILES):
            return candidate
    raise FileNotFoundError(
        f"No complete timestamped audit run found for {compartment} under {compartment_root}"
    )


# -------------------------- Resolve all three runs -------------------------
RUN_DIRS = {compartment: resolve_run_directory(compartment) for compartment in COMPARTMENTS}

review_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REVIEW_DIR = AUDIT_ROOT / "combined_normalized_endpoint_review_v1" / review_stamp
REVIEW_DIR.mkdir(parents=True, exist_ok=False)

run_inventory_rows = []
loaded = {}
for compartment, run_dir in RUN_DIRS.items():
    summary = json.loads((run_dir / "audit_summary.json").read_text(encoding="utf-8"))
    cohort = pd.read_csv(run_dir / "analysis_cohort.csv", dtype={"base_sample_id": str})
    matrix_qc = pd.read_csv(run_dir / "matrix_qc_summary.csv")
    patient_qc = pd.read_csv(run_dir / "patient_qc.csv", dtype={"base_sample_id": str})
    theta_map = pd.read_csv(run_dir / "theta_name_mapping.csv")
    theta_bmi = pd.read_csv(run_dir / "theta_bmi_associations.csv")
    near_overlap = read_csv_allow_empty(
        run_dir / "near_overlap_pairs_target.csv",
        columns=("analysis_celltype", "signature_a", "signature_b", "overlap_coefficient"),
    )
    concordance = pd.read_csv(run_dir / "score_variant_concordance.csv")
    slopes = pd.read_csv(run_dir / "descriptive_score_bmi_slopes.csv")
    scores = pd.read_csv(
        run_dir / "audit_scores_all_variants.csv.gz",
        dtype={"base_sample_id": str},
    )

    required_score_columns = {
        "base_sample_id", "CellType", "Signature", "feature", "variant", "score"
    }
    if not required_score_columns.issubset(scores.columns):
        raise ValueError(
            f"{compartment}: saved scores lack {sorted(required_score_columns - set(scores.columns))}"
        )
    observed_variants = set(scores["variant"].unique())
    expected_variants = set(RAW_VARIANTS + NORMALIZED_VARIANTS)
    if not expected_variants.issubset(observed_variants):
        raise ValueError(
            f"{compartment}: missing score variants {sorted(expected_variants - observed_variants)}"
        )
    if cohort["base_sample_id"].duplicated().any():
        raise ValueError(f"{compartment}: duplicate patient IDs in analysis_cohort.csv")
    if patient_qc.duplicated(["base_sample_id", "CellType"]).any():
        raise ValueError(f"{compartment}: duplicate patient/component rows in patient_qc.csv")

    loaded[compartment] = {
        "summary": summary,
        "cohort": cohort,
        "matrix_qc": matrix_qc,
        "patient_qc": patient_qc,
        "theta_map": theta_map,
        "theta_bmi": theta_bmi,
        "near_overlap": near_overlap,
        "concordance": concordance,
        "slopes": slopes,
        "scores": scores,
    }
    run_inventory_rows.append({
        "compartment": compartment,
        "run_directory": str(run_dir),
        "audit_version": summary.get("audit_version"),
        "overall_decision": summary.get("overall_decision"),
        "n_patients": int(cohort["base_sample_id"].nunique()),
        "n_celltypes": int(scores["CellType"].nunique()),
        "n_features": int(scores["feature"].nunique()),
    })

run_inventory = pd.DataFrame(run_inventory_rows)
atomic_to_csv(run_inventory, REVIEW_DIR / "selected_audit_runs.csv")


# ---------------- Component reliability and exact flag reasons ------------
reliability_rows = []
for compartment, data in loaded.items():
    patient_qc = data["patient_qc"].copy()
    matrix_qc = data["matrix_qc"].copy()

    for cell_type, group in patient_qc.groupby("CellType", sort=True):
        theta = pd.to_numeric(group["theta"], errors="coerce")
        theta_cv = pd.to_numeric(group["theta_cv"], errors="coerce")
        component_total = pd.to_numeric(group["component_total"], errors="coerce")
        zero_gene_fraction = pd.to_numeric(
            group["zero_fraction_nonmissing"], errors="coerce"
        )
        matrix_row = matrix_qc[matrix_qc["CellType"] == cell_type]
        if len(matrix_row) != 1:
            raise ValueError(
                f"{compartment}/{cell_type}: expected one matrix_qc row, found {len(matrix_row)}"
            )
        matrix_row = matrix_row.iloc[0]

        fraction_theta_below_005 = float((theta < 0.005).mean())
        fraction_theta_cv_above_1 = (
            float((theta_cv > 1.0).mean()) if theta_cv.notna().any() else np.nan
        )
        zero_total_patients = int((component_total <= 0).sum())
        flag_low_theta = fraction_theta_below_005 > LOW_THETA_FRACTION_LIMIT
        flag_high_theta_cv = (
            np.isfinite(fraction_theta_cv_above_1)
            and fraction_theta_cv_above_1 > HIGH_THETA_CV_FRACTION_LIMIT
        )
        flag_zero_total = zero_total_patients > 0
        flag_zero_heavy = finite_quantile(zero_gene_fraction, 0.50) > ZERO_HEAVY_GENE_FRACTION

        reasons = []
        if flag_low_theta:
            reasons.append("low_theta")
        if flag_high_theta_cv:
            reasons.append("high_theta_cv")
        if flag_zero_total:
            reasons.append("zero_component_total")
        if flag_zero_heavy:
            reasons.append("zero_heavy_expression")

        reliability_rows.append({
            "compartment": compartment,
            "CellType": cell_type,
            "audit_gate_flag": bool(flag_low_theta or flag_high_theta_cv or flag_zero_total),
            "extended_review_flag": bool(reasons),
            "review_reasons": ";".join(reasons) if reasons else "none",
            "flag_low_theta": bool(flag_low_theta),
            "flag_high_theta_cv": bool(flag_high_theta_cv),
            "flag_zero_component_total": bool(flag_zero_total),
            "flag_zero_heavy_expression": bool(flag_zero_heavy),
            "median_theta": finite_quantile(theta, 0.50),
            "fraction_theta_below_0_0001": float((theta < 0.0001).mean()),
            "fraction_theta_below_0_001": float((theta < 0.001).mean()),
            "fraction_theta_below_0_005": fraction_theta_below_005,
            "median_theta_cv": finite_quantile(theta_cv, 0.50),
            "p90_theta_cv": finite_quantile(theta_cv, 0.90),
            "fraction_theta_cv_above_0_5": (
                float((theta_cv > 0.5).mean()) if theta_cv.notna().any() else np.nan
            ),
            "fraction_theta_cv_above_1": fraction_theta_cv_above_1,
            "minimum_component_total": finite_quantile(component_total, 0.00),
            "p05_component_total": finite_quantile(component_total, 0.05),
            "median_component_total": finite_quantile(component_total, 0.50),
            "zero_total_patients": zero_total_patients,
            "median_zero_gene_fraction": finite_quantile(zero_gene_fraction, 0.50),
            "p90_zero_gene_fraction": finite_quantile(zero_gene_fraction, 0.90),
            "matrix_zero_fraction_nonmissing": float(matrix_row["zero_fraction_nonmissing"]),
        })

component_reliability = pd.DataFrame(reliability_rows)
component_reliability = component_reliability.sort_values(
    ["compartment", "extended_review_flag", "median_theta"],
    ascending=[True, False, True],
).reset_index(drop=True)

reliability_summary = component_reliability.groupby("compartment", sort=False).agg(
    n_components=("CellType", "nunique"),
    n_original_gate_flagged=("audit_gate_flag", "sum"),
    n_extended_review_flagged=("extended_review_flag", "sum"),
    n_low_theta=("flag_low_theta", "sum"),
    n_high_theta_cv=("flag_high_theta_cv", "sum"),
    n_zero_total=("flag_zero_component_total", "sum"),
    n_zero_heavy=("flag_zero_heavy_expression", "sum"),
).reset_index()

atomic_to_csv(component_reliability, REVIEW_DIR / "component_reliability_all.csv")
atomic_to_csv(
    component_reliability[component_reliability["extended_review_flag"]],
    REVIEW_DIR / "component_reliability_flagged.csv",
)
atomic_to_csv(reliability_summary, REVIEW_DIR / "component_reliability_summary.csv")


# ----------------------- Theta–BMI findings together ----------------------
theta_bmi_parts = []
for compartment, data in loaded.items():
    frame = data["theta_bmi"].copy()
    frame.insert(0, "compartment", compartment)
    modeled_theta = set(data["theta_map"]["theta_column"].dropna().astype(str))
    frame["modeled_component"] = frame["component"].astype(str).isin(modeled_theta)
    theta_bmi_parts.append(frame)

theta_bmi_all = pd.concat(theta_bmi_parts, ignore_index=True)
theta_bmi_significant = theta_bmi_all[
    pd.to_numeric(theta_bmi_all["q_value"], errors="coerce") < Q_LEVEL
].sort_values(["q_value", "compartment", "component"])
atomic_to_csv(theta_bmi_all, REVIEW_DIR / "theta_bmi_all_compartments.csv")
atomic_to_csv(
    theta_bmi_significant,
    REVIEW_DIR / "theta_bmi_q_lt_0_05.csv",
)


# -------- Per-feature raw/normalized and cap-3/cap-5 concordance ----------
cap_detail_parts = []
endpoint_detail_parts = []
for compartment, data in loaded.items():
    scores = data["scores"]
    slopes = data["slopes"]
    feature_meta = scores[["feature", "CellType", "Signature"]].drop_duplicates()
    if feature_meta["feature"].duplicated().any():
        raise ValueError(f"{compartment}: feature metadata is not one-to-one")

    score_wide = scores.pivot(
        index=["base_sample_id", "feature"], columns="variant", values="score"
    ).reset_index()
    slope_wide = slopes.pivot(index="feature", columns="variant", values="beta")

    for endpoint, cap3, cap5 in (
        ("raw_z", RAW_VARIANTS[0], RAW_VARIANTS[1]),
        ("within_component_logcpm_z", NORMALIZED_VARIANTS[0], NORMALIZED_VARIANTS[1]),
    ):
        if cap3 not in score_wide or cap5 not in score_wide:
            raise ValueError(f"{compartment}: missing {cap3} or {cap5}")
        for feature, group in score_wide.groupby("feature", sort=False):
            cap_detail_parts.append({
                "compartment": compartment,
                "feature": feature,
                "endpoint": endpoint,
                "cap3_vs_cap5_r": safe_correlation(group[cap3], group[cap5]),
                "slope_cap3": float(slope_wide.loc[feature, cap3]),
                "slope_cap5": float(slope_wide.loc[feature, cap5]),
                "slope_delta_cap5_minus_cap3": float(
                    slope_wide.loc[feature, cap5] - slope_wide.loc[feature, cap3]
                ),
            })

    for cap, raw_variant, normalized_variant in (
        (3, RAW_VARIANTS[0], NORMALIZED_VARIANTS[0]),
        (5, RAW_VARIANTS[1], NORMALIZED_VARIANTS[1]),
    ):
        for feature, group in score_wide.groupby("feature", sort=False):
            endpoint_detail_parts.append({
                "compartment": compartment,
                "feature": feature,
                "cap": cap,
                "raw_vs_normalized_r": safe_correlation(
                    group[raw_variant], group[normalized_variant]
                ),
                "raw_slope": float(slope_wide.loc[feature, raw_variant]),
                "normalized_slope": float(slope_wide.loc[feature, normalized_variant]),
                "normalized_minus_raw_slope": float(
                    slope_wide.loc[feature, normalized_variant]
                    - slope_wide.loc[feature, raw_variant]
                ),
            })

cap_concordance = pd.DataFrame(cap_detail_parts).merge(
    pd.concat(
        [
            data["scores"][["feature", "CellType", "Signature"]]
            .drop_duplicates()
            .assign(compartment=compartment)
            for compartment, data in loaded.items()
        ],
        ignore_index=True,
    ),
    on=["compartment", "feature"],
    how="left",
    validate="many_to_one",
)
endpoint_concordance = pd.DataFrame(endpoint_detail_parts).merge(
    pd.concat(
        [
            data["scores"][["feature", "CellType", "Signature"]]
            .drop_duplicates()
            .assign(compartment=compartment)
            for compartment, data in loaded.items()
        ],
        ignore_index=True,
    ),
    on=["compartment", "feature"],
    how="left",
    validate="many_to_one",
)

cap_summary_rows = []
for (compartment, endpoint), group in cap_concordance.groupby(
    ["compartment", "endpoint"], sort=False
):
    correlations = pd.to_numeric(group["cap3_vs_cap5_r"], errors="coerce")
    deltas = pd.to_numeric(group["slope_delta_cap5_minus_cap3"], errors="coerce").abs()
    cap_summary_rows.append({
        "compartment": compartment,
        "endpoint": endpoint,
        "n_features": int(group["feature"].nunique()),
        "minimum_cap3_vs_cap5_r": finite_quantile(correlations, 0.00),
        "p01_cap3_vs_cap5_r": finite_quantile(correlations, 0.01),
        "median_cap3_vs_cap5_r": finite_quantile(correlations, 0.50),
        "features_r_below_0_99": int((correlations < 0.99).sum()),
        "maximum_abs_slope_delta": finite_quantile(deltas, 1.00),
        "p99_abs_slope_delta": finite_quantile(deltas, 0.99),
        "median_abs_slope_delta": finite_quantile(deltas, 0.50),
        "features_abs_slope_delta_above_0_05": int((deltas > 0.05).sum()),
    })
cap_summary = pd.DataFrame(cap_summary_rows)

endpoint_summary_rows = []
for (compartment, cap), group in endpoint_concordance.groupby(
    ["compartment", "cap"], sort=False
):
    correlations = pd.to_numeric(group["raw_vs_normalized_r"], errors="coerce")
    deltas = pd.to_numeric(group["normalized_minus_raw_slope"], errors="coerce").abs()
    endpoint_summary_rows.append({
        "compartment": compartment,
        "cap": int(cap),
        "n_features": int(group["feature"].nunique()),
        "minimum_raw_vs_normalized_r": finite_quantile(correlations, 0.00),
        "p05_raw_vs_normalized_r": finite_quantile(correlations, 0.05),
        "median_raw_vs_normalized_r": finite_quantile(correlations, 0.50),
        "maximum_abs_slope_change": finite_quantile(deltas, 1.00),
        "p95_abs_slope_change": finite_quantile(deltas, 0.95),
    })
endpoint_summary = pd.DataFrame(endpoint_summary_rows)

atomic_to_csv(cap_concordance, REVIEW_DIR / "cap3_vs_cap5_per_feature.csv")
atomic_to_csv(cap_summary, REVIEW_DIR / "cap3_vs_cap5_summary.csv")
atomic_to_csv(endpoint_concordance, REVIEW_DIR / "raw_vs_normalized_per_feature.csv")
atomic_to_csv(endpoint_summary, REVIEW_DIR / "raw_vs_normalized_summary.csv")


# -------- Dependence on normalized endpoints, with exact denominators -----
dependence_pair_rows = []
dependence_celltype_rows = []
cluster_parts = []

for compartment, data in loaded.items():
    scores = data["scores"]
    cohort = data["cohort"].copy().set_index("base_sample_id")
    patient_qc = data["patient_qc"].copy()
    reliability_lookup = component_reliability[
        component_reliability["compartment"] == compartment
    ].set_index("CellType")

    required_design_columns = ["bmi_z", "age_z", "sex_centered"]
    missing_design = [column for column in required_design_columns if column not in cohort]
    if missing_design:
        raise ValueError(f"{compartment}: cohort lacks design columns {missing_design}")

    near_overlap = data["near_overlap"]
    near_lookup = set()
    if not near_overlap.empty and {
        "analysis_celltype", "signature_a", "signature_b"
    }.issubset(near_overlap.columns):
        for row in near_overlap.itertuples(index=False):
            near_lookup.add(
                (
                    str(row.analysis_celltype),
                    *sorted((str(row.signature_a), str(row.signature_b))),
                )
            )

    for variant in NORMALIZED_VARIANTS:
        variant_scores = scores[scores["variant"] == variant].copy()
        for cell_type, group in variant_scores.groupby("CellType", sort=True):
            meta = group[["feature", "Signature"]].drop_duplicates()
            if meta["feature"].duplicated().any():
                raise ValueError(
                    f"{compartment}/{variant}/{cell_type}: feature-to-signature mapping is not unique"
                )
            signature_by_feature = meta.set_index("feature")["Signature"].to_dict()
            pivot = group.pivot(
                index="base_sample_id", columns="feature", values="score"
            ).sort_index()
            theta = patient_qc[patient_qc["CellType"] == cell_type].set_index(
                "base_sample_id"
            )["theta"].rename("theta")
            design = cohort[required_design_columns].join(theta, how="inner")
            common_ids = pivot.index.intersection(design.index)
            Y = pivot.loc[common_ids]
            D = design.loc[common_ids]
            valid = Y.notna().all(axis=1) & D.notna().all(axis=1)
            Y = Y.loc[valid]
            D = D.loc[valid]
            if len(Y) < 10:
                raise ValueError(
                    f"{compartment}/{variant}/{cell_type}: only {len(Y)} complete patients"
                )

            features = list(Y.columns)
            values = Y.to_numpy(dtype=float)
            raw_corr = correlation_matrix(values)
            covariate_design = np.column_stack(
                [np.ones(len(D)), D[required_design_columns].to_numpy(dtype=float)]
            )
            covariate_theta_design = np.column_stack(
                [
                    np.ones(len(D)),
                    D[required_design_columns + ["theta"]].to_numpy(dtype=float),
                ]
            )
            covariate_residual_corr = correlation_matrix(
                residualize_matrix(values, covariate_design)
            )
            covariate_theta_residual_corr = correlation_matrix(
                residualize_matrix(values, covariate_theta_design)
            )

            possible_pairs = len(features) * (len(features) - 1) // 2
            high_counts = {
                "raw": 0,
                "residual_bmi_age_sex": 0,
                "residual_bmi_age_sex_theta": 0,
            }
            residual_theta_overlap = 0
            residual_theta_nonoverlap = 0
            edges_by_kind = {key: [] for key in high_counts}

            for left in range(len(features)):
                for right in range(left + 1, len(features)):
                    feature_a, feature_b = features[left], features[right]
                    signature_a = str(signature_by_feature[feature_a])
                    signature_b = str(signature_by_feature[feature_b])
                    r_raw = float(raw_corr[left, right])
                    r_cov = float(covariate_residual_corr[left, right])
                    r_cov_theta = float(covariate_theta_residual_corr[left, right])
                    values_by_kind = {
                        "raw": r_raw,
                        "residual_bmi_age_sex": r_cov,
                        "residual_bmi_age_sex_theta": r_cov_theta,
                    }
                    for kind, correlation in values_by_kind.items():
                        if np.isfinite(correlation) and abs(correlation) >= CORRELATION_THRESHOLD:
                            high_counts[kind] += 1
                            edges_by_kind[kind].append((feature_a, feature_b))

                    gene_overlap = (
                        str(cell_type), *sorted((signature_a, signature_b))
                    ) in near_lookup
                    if (
                        np.isfinite(r_cov_theta)
                        and abs(r_cov_theta) >= CORRELATION_THRESHOLD
                    ):
                        if gene_overlap:
                            residual_theta_overlap += 1
                        else:
                            residual_theta_nonoverlap += 1

                    if any(
                        np.isfinite(correlation)
                        and abs(correlation) >= CORRELATION_THRESHOLD
                        for correlation in values_by_kind.values()
                    ):
                        dependence_pair_rows.append({
                            "compartment": compartment,
                            "variant": variant,
                            "CellType": cell_type,
                            "feature_a": feature_a,
                            "feature_b": feature_b,
                            "signature_a": signature_a,
                            "signature_b": signature_b,
                            "normalized_raw_r": r_raw,
                            "residual_r_after_bmi_age_sex": r_cov,
                            "residual_r_after_bmi_age_sex_theta": r_cov_theta,
                            "gene_overlap_gt_0_50": bool(gene_overlap),
                        })

            for kind, edges in edges_by_kind.items():
                clusters = connected_components(
                    features,
                    edges,
                    prefix=(
                        f"{compartment}_{variant}_{cell_type}_{kind}"
                        .replace(" ", "_")
                        .replace("/", "_")
                    ),
                )
                clusters.insert(0, "compartment", compartment)
                clusters.insert(1, "variant", variant)
                clusters.insert(2, "CellType", cell_type)
                clusters.insert(3, "correlation_kind", kind)
                clusters["Signature"] = clusters["feature"].map(signature_by_feature)
                cluster_parts.append(clusters)

            dependence_celltype_rows.append({
                "compartment": compartment,
                "variant": variant,
                "CellType": cell_type,
                "component_audit_gate_flag": bool(
                    reliability_lookup.loc[cell_type, "audit_gate_flag"]
                ),
                "component_extended_review_flag": bool(
                    reliability_lookup.loc[cell_type, "extended_review_flag"]
                ),
                "component_review_reasons": reliability_lookup.loc[
                    cell_type, "review_reasons"
                ],
                "n_patients": int(len(Y)),
                "n_features": int(len(features)),
                "n_possible_pairs": int(possible_pairs),
                "n_high_raw": int(high_counts["raw"]),
                "proportion_high_raw": (
                    high_counts["raw"] / possible_pairs if possible_pairs else np.nan
                ),
                "n_high_residual_bmi_age_sex": int(
                    high_counts["residual_bmi_age_sex"]
                ),
                "proportion_high_residual_bmi_age_sex": (
                    high_counts["residual_bmi_age_sex"] / possible_pairs
                    if possible_pairs else np.nan
                ),
                "n_high_residual_bmi_age_sex_theta": int(
                    high_counts["residual_bmi_age_sex_theta"]
                ),
                "proportion_high_residual_bmi_age_sex_theta": (
                    high_counts["residual_bmi_age_sex_theta"] / possible_pairs
                    if possible_pairs else np.nan
                ),
                "n_high_residual_theta_with_gene_overlap": int(
                    residual_theta_overlap
                ),
                "n_high_residual_theta_without_gene_overlap": int(
                    residual_theta_nonoverlap
                ),
            })

normalized_dependence_pairs = pd.DataFrame(dependence_pair_rows)
normalized_dependence_by_celltype = pd.DataFrame(dependence_celltype_rows)
normalized_clusters = pd.concat(cluster_parts, ignore_index=True)


def dependence_scope_summary(frame: pd.DataFrame, scope: str) -> list[dict]:
    rows = []
    for (compartment, variant), group in frame.groupby(
        ["compartment", "variant"], sort=False
    ):
        if scope == "audit_clear_components_only":
            group = group[~group["component_audit_gate_flag"]]
        possible = int(group["n_possible_pairs"].sum())
        n_raw = int(group["n_high_raw"].sum())
        n_cov = int(group["n_high_residual_bmi_age_sex"].sum())
        n_cov_theta = int(group["n_high_residual_bmi_age_sex_theta"].sum())
        cluster_subset = normalized_clusters[
            (normalized_clusters["compartment"] == compartment)
            & (normalized_clusters["variant"] == variant)
            & (
                normalized_clusters["correlation_kind"]
                == "residual_bmi_age_sex_theta"
            )
            & (normalized_clusters["CellType"].isin(group["CellType"]))
            & (~normalized_clusters["is_singleton"])
        ]
        rows.append({
            "compartment": compartment,
            "variant": variant,
            "scope": scope,
            "n_components": int(group["CellType"].nunique()),
            "n_features": int(group["n_features"].sum()),
            "n_possible_pairs": possible,
            "n_high_raw": n_raw,
            "proportion_high_raw": n_raw / possible if possible else np.nan,
            "n_high_residual_bmi_age_sex": n_cov,
            "proportion_high_residual_bmi_age_sex": (
                n_cov / possible if possible else np.nan
            ),
            "n_high_residual_bmi_age_sex_theta": n_cov_theta,
            "proportion_high_residual_bmi_age_sex_theta": (
                n_cov_theta / possible if possible else np.nan
            ),
            "n_residual_theta_non_singleton_clusters": int(
                cluster_subset["cluster_id"].nunique()
            ),
            "largest_residual_theta_cluster": int(
                cluster_subset["cluster_size"].max()
            ) if not cluster_subset.empty else 1,
            "n_high_residual_theta_with_gene_overlap": int(
                group["n_high_residual_theta_with_gene_overlap"].sum()
            ),
            "n_high_residual_theta_without_gene_overlap": int(
                group["n_high_residual_theta_without_gene_overlap"].sum()
            ),
        })
    return rows


normalized_dependence_summary = pd.DataFrame(
    dependence_scope_summary(normalized_dependence_by_celltype, "all_components")
    + dependence_scope_summary(
        normalized_dependence_by_celltype, "audit_clear_components_only"
    )
)

atomic_to_csv(
    normalized_dependence_pairs,
    REVIEW_DIR / "normalized_high_correlation_pairs.csv",
)
atomic_to_csv(
    normalized_dependence_by_celltype,
    REVIEW_DIR / "normalized_dependence_by_celltype.csv",
)
atomic_to_csv(
    normalized_dependence_summary,
    REVIEW_DIR / "normalized_dependence_summary.csv",
)
atomic_to_csv(
    normalized_clusters,
    REVIEW_DIR / "normalized_correlation_cluster_membership.csv",
)


# -------------------------- Compact review report -------------------------
manifest = {
    "review_version": "combined-normalized-endpoint-review-v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "review_directory": str(REVIEW_DIR),
    "selected_runs": {key: str(value) for key, value in RUN_DIRS.items()},
    "thresholds": {
        "correlation_abs": CORRELATION_THRESHOLD,
        "low_theta": LOW_THETA_THRESHOLD,
        "low_theta_fraction_limit": LOW_THETA_FRACTION_LIMIT,
        "high_theta_cv": HIGH_THETA_CV_THRESHOLD,
        "high_theta_cv_fraction_limit": HIGH_THETA_CV_FRACTION_LIMIT,
        "zero_heavy_gene_fraction": ZERO_HEAVY_GENE_FRACTION,
        "theta_bmi_q": Q_LEVEL,
    },
    "notes": [
        "No model was fitted.",
        "audit_gate_flag reproduces the original low-theta/high-CV/zero-total rule.",
        "extended_review_flag additionally identifies zero-heavy expression.",
        "No flagged component is automatically excluded by this cell.",
        "Normalized dependence is reported for cap 3 and cap 5 with exact pair denominators.",
    ],
}
atomic_write_text(
    json.dumps(manifest, indent=2),
    REVIEW_DIR / "combined_review_manifest.json",
)

report_lines = [
    "# Combined BayesPrism normalized-endpoint review",
    "",
    "This addendum fitted no model and excluded no component.",
    "",
    f"- Selected audit runs: {len(run_inventory)}/3",
    f"- Components reviewed: {len(component_reliability)}",
    f"- Original-gate flagged components: {int(component_reliability['audit_gate_flag'].sum())}",
    f"- Extended-review components: {int(component_reliability['extended_review_flag'].sum())}",
    f"- Theta–BMI components with q < {Q_LEVEL}: {len(theta_bmi_significant)}",
    f"- Normalized variants audited: {', '.join(NORMALIZED_VARIANTS)}",
    "",
    "Use the CSV tables in this directory to choose the reliability rule, clipping cap, and correlation-cluster strategy before fitting the replacement model.",
]
report_text = "\n".join(report_lines) + "\n"
atomic_write_text(report_text, REVIEW_DIR / "COMBINED_REVIEW.md")


# ------------------------------- Display ---------------------------------
display(Markdown(report_text))

print("\nSELECTED AUDIT RUNS")
display(run_inventory)

print("\nCOMPONENT RELIABILITY SUMMARY")
display(reliability_summary)

print("\nFLAGGED COMPONENTS AND EXACT REASONS")
display(
    component_reliability.loc[
        component_reliability["extended_review_flag"],
        [
            "compartment", "CellType", "review_reasons", "median_theta",
            "fraction_theta_below_0_001", "fraction_theta_below_0_005",
            "median_theta_cv", "p90_theta_cv", "fraction_theta_cv_above_1",
            "p05_component_total", "zero_total_patients",
            "median_zero_gene_fraction", "p90_zero_gene_fraction",
        ],
    ]
)

print("\nTHETA–BMI COMPONENTS WITH q < 0.05")
if theta_bmi_significant.empty:
    print("None")
else:
    display(
        theta_bmi_significant[
            [
                "compartment", "component", "modeled_component", "beta", "se",
                "p_value", "q_value", "median_theta", "fraction_below_low_theta",
                "median_theta_cv",
            ]
        ]
    )

print("\nCAP 3 VERSUS CAP 5 — RAW AND NORMALIZED ENDPOINTS")
display(cap_summary)

print("\nRAW VERSUS NORMALIZED ENDPOINT")
display(endpoint_summary)

print("\nNORMALIZED-ENDPOINT DEPENDENCE WITH EXACT DENOMINATORS")
display(normalized_dependence_summary)

print("\nMOST DEPENDENT NORMALIZED PAIRS AFTER BMI/AGE/SEX/THETA PROJECTION")
if normalized_dependence_pairs.empty:
    print("None")
else:
    top_pairs = normalized_dependence_pairs.assign(
        abs_residual_r=lambda frame: frame[
            "residual_r_after_bmi_age_sex_theta"
        ].abs()
    ).sort_values("abs_residual_r", ascending=False).head(30)
    display(
        top_pairs[
            [
                "compartment", "variant", "CellType", "signature_a", "signature_b",
                "normalized_raw_r", "residual_r_after_bmi_age_sex",
                "residual_r_after_bmi_age_sex_theta", "gene_overlap_gt_0_50",
            ]
        ]
    )

print("\nSaved combined review to:", REVIEW_DIR)
print(
    "Send the five displayed summary tables (run inventory, flagged components, "
    "theta–BMI, cap summary, and normalized-dependence summary)."
)


In [ ]:


# BayesPrism publication PCA: tiered primary versus strict-QC sensitivity




from __future__ import annotations

import hashlib
import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.linalg import subspace_angles
from scipy.spatial.distance import pdist
from scipy.stats import chi2, spearmanr, t as student_t
from sklearn.covariance import MinCovDet
from sklearn.decomposition import PCA

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        print(value)


# ----------------------------- Configuration -----------------------------
AUDIT_ROOT = Path(
    globals().get(
        "PCA_AUDIT_ROOT",
        "/content/drive/MyDrive/new_dysfunction/"
        "bayesprism_initial_issue_verification_v1",
    )
)

# Normally leave this as None. It selects the latest complete combined review.
# To freeze an exact run, assign that timestamped combined-review directory.
COMBINED_REVIEW_DIR_OVERRIDE = globals().get("PCA_COMBINED_REVIEW_DIR", None)

COMPARTMENTS = ("immune_course", "immune_fine", "non_immune")
DISPLAY_NAMES = {
    "immune_course": "Immune coarse",
    "immune_fine": "Immune fine",
    "non_immune": "Non-immune",
}
ANALYSIS_FAMILIES = {
    "immune_course": "broad_primary",
    "non_immune": "broad_primary",
    # Fine immune is nested within coarse immune and is not an independent family.
    "immune_fine": "fine_resolution_secondary",
}

PRIMARY_VARIANT = "within_component_logcpm_z_cap5"
CLUSTER_KIND = "residual_bmi_age_sex_theta"
ZERO_HEAVY_THRESHOLD = 0.20
N_PCS_SAVE = 10
N_PCS_TEST = 5
N_PCS_OUTLIER = 5
OUTLIER_ALPHA = 0.001             # review flag only; never automatic removal
SEED = 20260904

# Freeze the independently reviewed v1.3 audit. Any mismatch stops execution.
EXPECTED = {
    "immune_course": dict(all=8, tiered=8, strict=6, borderline=2, excluded=0,
                          tiered_features=306, strict_features=232),
    "immune_fine": dict(all=22, tiered=14, strict=12, borderline=2, excluded=8,
                        tiered_features=496, strict_features=430),
    "non_immune": dict(all=17, tiered=6, strict=4, borderline=2, excluded=11,
                       tiered_features=157, strict_features=112),
}


# ------------------------------- Helpers ---------------------------------
def atomic_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    temporary.replace(path)


def atomic_text(value: str, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(value, encoding="utf-8")
    temporary.replace(path)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def bh_adjust(values) -> np.ndarray:
    p = np.asarray(values, dtype=float)
    q = np.full(len(p), np.nan)
    finite = np.isfinite(p)
    if not finite.any():
        return q
    observed = p[finite]
    order = np.argsort(observed)
    adjusted_ordered = observed[order] * len(observed) / np.arange(1, len(observed) + 1)
    adjusted_ordered = np.minimum.accumulate(adjusted_ordered[::-1])[::-1]
    adjusted = np.empty(len(observed))
    adjusted[order] = np.clip(adjusted_ordered, 0, 1)
    q[finite] = adjusted
    return q


def zscore(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    sd = values.std(ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return pd.Series(np.nan, index=series.index)
    return (values - values.mean()) / sd


def latest_complete_review() -> Path:
    required = (
        "combined_review_manifest.json",
        "component_reliability_all.csv",
        "normalized_correlation_cluster_membership.csv",
        "theta_bmi_all_compartments.csv",
    )
    if COMBINED_REVIEW_DIR_OVERRIDE:
        candidate = Path(COMBINED_REVIEW_DIR_OVERRIDE)
        missing = [name for name in required if not (candidate / name).exists()]
        if missing:
            raise FileNotFoundError(f"Review override is missing {missing}")
        return candidate

    parent = AUDIT_ROOT / "combined_normalized_endpoint_review_v1"
    if not parent.exists():
        raise FileNotFoundError(f"Combined review root not found: {parent}")
    candidates = sorted(
        (path for path in parent.iterdir() if path.is_dir()),
        key=lambda path: path.name,
        reverse=True,
    )
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f"No complete combined review found under {parent}")


def derive_tiers(reliability: pd.DataFrame) -> pd.DataFrame:
    required = {
        "compartment", "CellType", "flag_low_theta", "flag_high_theta_cv",
        "flag_zero_component_total", "flag_zero_heavy_expression",
        "zero_total_patients", "median_zero_gene_fraction",
    }
    missing = required - set(reliability.columns)
    if missing:
        raise ValueError(f"Reliability table lacks {sorted(missing)}")

    result = reliability.copy()
    # Recompute from numeric evidence rather than trusting the report decision.
    result["exclude_zero_heavy"] = (
        pd.to_numeric(result["zero_total_patients"], errors="coerce").gt(0)
        | pd.to_numeric(result["median_zero_gene_fraction"], errors="coerce").gt(
            ZERO_HEAVY_THRESHOLD
        )
    )
    result["tiered_primary_keep"] = ~result["exclude_zero_heavy"]
    result["strict_qc_keep"] = ~(
        result["flag_low_theta"].astype(bool)
        | result["flag_high_theta_cv"].astype(bool)
        | result["flag_zero_component_total"].astype(bool)
        | result["flag_zero_heavy_expression"].astype(bool)
    )
    result["borderline_supported"] = (
        result["tiered_primary_keep"] & ~result["strict_qc_keep"]
    )
    result["analysis_tier"] = np.select(
        [result["exclude_zero_heavy"], result["borderline_supported"]],
        ["exploratory_zero_heavy", "borderline_supported"],
        default="strict_qc_clear",
    )
    result["analysis_family"] = result["compartment"].map(ANALYSIS_FAMILIES)

    if result.duplicated(["compartment", "CellType"]).any():
        raise ValueError("Duplicate compartment/CellType reliability rows")
    for compartment, expected in EXPECTED.items():
        local = result[result["compartment"].eq(compartment)]
        observed = dict(
            all=len(local),
            tiered=int(local["tiered_primary_keep"].sum()),
            strict=int(local["strict_qc_keep"].sum()),
            borderline=int(local["borderline_supported"].sum()),
            excluded=int(local["exclude_zero_heavy"].sum()),
        )
        target = {key: expected[key] for key in observed}
        if observed != target:
            raise ValueError(
                f"{compartment}: reviewed tier counts changed; "
                f"expected {target}, observed {observed}."
            )
    return result


def hc3_bmi_test(frame: pd.DataFrame, outcome: str) -> dict:
    work = frame[[outcome, "bmi_z", "age_z", "sex_centered"]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    y = zscore(work[outcome]).to_numpy(float)
    X = np.column_stack([
        np.ones(len(work)), work["bmi_z"], work["age_z"], work["sex_centered"]
    ]).astype(float)
    bread = np.linalg.pinv(X.T @ X)
    beta = bread @ X.T @ y
    residual = y - X @ beta
    leverage = np.einsum("ij,jk,ik->i", X, bread, X)
    scaled = residual / np.clip(1 - leverage, 1e-8, None)
    covariance = bread @ (X.T @ ((scaled ** 2)[:, None] * X)) @ bread
    se = float(np.sqrt(max(covariance[1, 1], 0)))
    dof = len(work) - np.linalg.matrix_rank(X)
    statistic = float(beta[1] / se) if se > 0 else np.nan
    p_value = float(2 * student_t.sf(abs(statistic), dof)) if dof > 0 else np.nan
    return dict(n=len(work), beta_bmi=float(beta[1]), se_hc3=se, p_value=p_value)


def ellipse_xy(x, y, probability=0.95):
    values = np.column_stack([x, y]).astype(float)
    values = values[np.isfinite(values).all(axis=1)]
    if len(values) < 5:
        return None
    covariance = np.cov(values, rowvar=False, ddof=1)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    if not np.isfinite(eigenvalues).all() or (eigenvalues <= 0).any():
        return None
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.linspace(0, 2 * np.pi, 240)
    unit_circle = np.vstack([np.cos(angle), np.sin(angle)])
    scale = math.sqrt(float(chi2.ppf(probability, 2)))
    ellipse = values.mean(axis=0)[:, None] + (
        eigenvectors @ np.diag(np.sqrt(eigenvalues)) @ unit_circle * scale
    )
    return ellipse[0], ellipse[1]


def save_figure(fig, stem: Path) -> None:
    for suffix, options in (("png", {"dpi": 600}), ("pdf", {}), ("svg", {})):
        destination = stem.with_suffix(f".{suffix}")
        temporary = destination.with_name(destination.stem + ".tmp" + destination.suffix)
        fig.savefig(temporary, bbox_inches="tight", facecolor="white", **options)
        temporary.replace(destination)


def build_weighted_matrix(scores, cohort, clusters, compartment, celltypes):
    local = scores[
        scores["variant"].eq(PRIMARY_VARIANT)
        & scores["CellType"].isin(celltypes)
    ].copy()
    if local.empty or local.duplicated(["base_sample_id", "feature"]).any():
        raise ValueError(f"{compartment}: empty or duplicated PCA score input")

    metadata = local[["feature", "CellType", "Signature"]].drop_duplicates()
    if metadata["feature"].duplicated().any():
        raise ValueError(f"{compartment}: feature metadata is not one-to-one")
    matrix = local.pivot(index="base_sample_id", columns="feature", values="score")
    matrix.index = matrix.index.astype(str)
    matrix = matrix.reindex(cohort["base_sample_id"].astype(str))
    if matrix.isna().any().any():
        raise ValueError(f"{compartment}: incomplete patient-feature PCA matrix")
    standard = (matrix - matrix.mean()) / matrix.std(ddof=1)
    if not np.isfinite(standard.to_numpy()).all():
        raise ValueError(f"{compartment}: constant or nonfinite PCA feature")

    cluster_map = clusters[
        clusters["compartment"].eq(compartment)
        & clusters["variant"].eq(PRIMARY_VARIANT)
        & clusters["correlation_kind"].eq(CLUSTER_KIND)
    ][["feature", "cluster_id"]].drop_duplicates()
    if cluster_map["feature"].duplicated().any():
        raise ValueError(f"{compartment}: feature occurs in multiple clusters")
    metadata = metadata.merge(cluster_map, on="feature", how="left", validate="one_to_one")
    fallback = compartment + "__singleton__" + metadata["feature"].astype(str)
    metadata["cluster_id"] = metadata["cluster_id"].where(metadata["cluster_id"].notna(), fallback)
    metadata["cluster_size_retained"] = metadata.groupby(
        ["CellType", "cluster_id"]
    )["feature"].transform("size")
    metadata["clusters_in_celltype"] = metadata.groupby("CellType")[
        "cluster_id"
    ].transform("nunique")
    metadata["pca_weight"] = 1 / np.sqrt(
        metadata["cluster_size_retained"] * metadata["clusters_in_celltype"]
    )
    weight_check = metadata.groupby("CellType")["pca_weight"].apply(
        lambda values: np.square(values).sum()
    )
    if not np.allclose(weight_check, 1.0):
        raise AssertionError(f"{compartment}: invalid block weights")
    metadata = metadata.set_index("feature").loc[standard.columns].reset_index()
    weights = metadata.set_index("feature")["pca_weight"]
    return standard, standard * weights, metadata


def run_pca(compartment, analysis_set, scores, cohort, clusters, celltypes):
    standard, weighted, metadata = build_weighted_matrix(
        scores, cohort, clusters, compartment, celltypes
    )
    n_pcs = min(N_PCS_SAVE, weighted.shape[0] - 1, weighted.shape[1])
    pca = PCA(n_components=n_pcs, svd_solver="full")
    values = pca.fit_transform(weighted)
    pc_names = [f"PC{index}" for index in range(1, n_pcs + 1)]

    patients = pd.DataFrame(values, index=weighted.index, columns=pc_names).reset_index()
    patients = patients.rename(columns={patients.columns[0]: "base_sample_id"})
    patients.insert(0, "compartment", compartment)
    patients.insert(1, "analysis_set", analysis_set)
    patients = patients.merge(cohort, on="base_sample_id", how="left", validate="one_to_one")

    explained = pd.DataFrame({
        "compartment": compartment,
        "analysis_set": analysis_set,
        "PC": pc_names,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    })

    correlations = np.corrcoef(standard.to_numpy().T, values.T)
    correlations = correlations[:standard.shape[1], standard.shape[1]:]
    loading_parts = []
    for number, pc_name in enumerate(pc_names):
        part = metadata.copy()
        part.insert(0, "compartment", compartment)
        part.insert(1, "analysis_set", analysis_set)
        part["PC"] = pc_name
        part["axis_coefficient"] = pca.components_[number]
        part["feature_pc_correlation"] = correlations[:, number]
        part["squared_axis_contribution"] = np.square(pca.components_[number])
        loading_parts.append(part)
    loadings = pd.concat(loading_parts, ignore_index=True)
    contributions = loadings.groupby(
        ["compartment", "analysis_set", "PC", "CellType"], as_index=False
    )["squared_axis_contribution"].sum()
    contributions["percent_axis_contribution"] = 100 * contributions["squared_axis_contribution"]

    tests = []
    for pc_name in pc_names[:N_PCS_TEST]:
        tests.append({
            "compartment": compartment,
            "analysis_set": analysis_set,
            "PC": pc_name,
            **hc3_bmi_test(patients, pc_name),
        })
    tests = pd.DataFrame(tests)
    tests["q_within_compartment_set"] = bh_adjust(tests["p_value"])

    k = min(N_PCS_OUTLIER, n_pcs)
    outlier_input = values[:, :k] / np.sqrt(pca.explained_variance_[:k])
    method = "minimum_covariance_determinant"
    try:
        distance = MinCovDet(random_state=SEED, support_fraction=0.75).fit(
            outlier_input
        ).mahalanobis(outlier_input)
    except Exception as error:
        warnings.warn(f"Robust outlier covariance failed: {error}")
        method = "classical_covariance_fallback"
        centered = outlier_input - outlier_input.mean(axis=0)
        inverse = np.linalg.pinv(np.cov(centered, rowvar=False, ddof=1))
        distance = np.einsum("ij,jk,ik->i", centered, inverse, centered)
    threshold = float(chi2.ppf(1 - OUTLIER_ALPHA, k))
    outliers = patients[["base_sample_id", "bmi", "bmi_z", "bmi_group"]].copy()
    outliers.insert(0, "compartment", compartment)
    outliers.insert(1, "analysis_set", analysis_set)
    outliers["method"] = method
    outliers["pca_dimensions"] = k
    outliers["squared_distance"] = distance
    outliers["review_threshold"] = threshold
    outliers["review_flag"] = distance > threshold

    return dict(
        standard=standard,
        weighted=weighted,
        pca=pca,
        patient_scores=patients,
        explained=explained,
        loadings=loadings,
        contributions=contributions,
        tests=tests,
        outliers=outliers,
        patient_basis=values / np.clip(pca.singular_values_, 1e-15, None),
    )


# --------------------------- Resolve saved inputs -------------------------
REVIEW_DIR = latest_complete_review()
manifest_path = REVIEW_DIR / "combined_review_manifest.json"
review_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
run_dirs = {
    compartment: Path(review_manifest["selected_runs"][compartment])
    for compartment in COMPARTMENTS
}
for compartment, run_dir in run_dirs.items():
    missing = [
        name for name in ("analysis_cohort.csv", "audit_scores_all_variants.csv.gz")
        if not (run_dir / name).exists()
    ]
    if missing:
        raise FileNotFoundError(f"{compartment}: selected audit lacks {missing}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = AUDIT_ROOT / "publication_pca_tiered_qc_v1" / stamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

reliability_path = REVIEW_DIR / "component_reliability_all.csv"
clusters_path = REVIEW_DIR / "normalized_correlation_cluster_membership.csv"
theta_path = REVIEW_DIR / "theta_bmi_all_compartments.csv"
tiers = derive_tiers(pd.read_csv(reliability_path))
clusters = pd.read_csv(clusters_path)
theta_bmi = pd.read_csv(theta_path)
atomic_csv(tiers, OUTPUT_DIR / "component_inclusion_tiers.csv")

# Recalculate theta multiplicity across all compartments; original q is within-compartment.
theta_bmi["q_global_all_components"] = bh_adjust(theta_bmi["p_value"])
modeled = theta_bmi["modeled_component"].astype(bool)
theta_bmi["q_global_modeled_components"] = np.nan
theta_bmi.loc[modeled, "q_global_modeled_components"] = bh_adjust(
    theta_bmi.loc[modeled, "p_value"]
)
atomic_csv(theta_bmi, OUTPUT_DIR / "theta_bmi_multiplicity_recalculated.csv")


# -------------------------------- Run PCA --------------------------------
results = {}
input_counts = []
input_hashes = []
for compartment in COMPARTMENTS:
    run_dir = run_dirs[compartment]
    cohort_path = run_dir / "analysis_cohort.csv"
    scores_path = run_dir / "audit_scores_all_variants.csv.gz"
    cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
    scores = pd.read_csv(scores_path, dtype={"base_sample_id": str})
    required_cohort = {"base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"}
    required_scores = {"base_sample_id", "CellType", "Signature", "feature", "variant", "score"}
    if not required_cohort.issubset(cohort.columns):
        raise ValueError(f"{compartment}: incomplete cohort columns")
    if not required_scores.issubset(scores.columns):
        raise ValueError(f"{compartment}: incomplete score columns")
    if cohort["base_sample_id"].nunique() != 127 or cohort["base_sample_id"].duplicated().any():
        raise ValueError(f"{compartment}: expected 127 unique patients")
    expected_groups = {"normal_weight": 52, "overweight": 57, "obese": 18}
    if cohort["bmi_group"].value_counts().to_dict() != expected_groups:
        raise ValueError(f"{compartment}: BMI cohort counts changed")

    local_tiers = tiers[tiers["compartment"].eq(compartment)]
    sets = {
        "tiered_primary": sorted(local_tiers.loc[local_tiers["tiered_primary_keep"], "CellType"]),
        "strict_qc": sorted(local_tiers.loc[local_tiers["strict_qc_keep"], "CellType"]),
    }
    results[compartment] = {}
    for set_name, celltypes in sets.items():
        result = run_pca(compartment, set_name, scores, cohort, clusters, celltypes)
        results[compartment][set_name] = result
        n_features = result["weighted"].shape[1]
        expected_features = EXPECTED[compartment][
            "tiered_features" if set_name == "tiered_primary" else "strict_features"
        ]
        if n_features != expected_features:
            raise ValueError(
                f"{compartment}/{set_name}: expected {expected_features} features, found {n_features}"
            )
        input_counts.append(dict(
            compartment=compartment,
            analysis_family=ANALYSIS_FAMILIES[compartment],
            analysis_set=set_name,
            n_patients=result["weighted"].shape[0],
            n_celltypes=len(celltypes),
            n_features=n_features,
        ))
    input_hashes.extend([
        dict(compartment=compartment, input="cohort", path=str(cohort_path), sha256=file_sha256(cohort_path)),
        dict(compartment=compartment, input="scores", path=str(scores_path), sha256=file_sha256(scores_path)),
    ])


# ------------------------- Stability comparison --------------------------
stability_rows = []
for compartment in COMPARTMENTS:
    tiered = results[compartment]["tiered_primary"]
    strict = results[compartment]["strict_qc"]
    distance_rho = float(spearmanr(
        pdist(tiered["weighted"]), pdist(strict["weighted"])
    )[0])
    cosine = {}
    for requested_k in (2, 5):
        k = min(requested_k, tiered["patient_basis"].shape[1], strict["patient_basis"].shape[1])
        angles = subspace_angles(tiered["patient_basis"][:, :k], strict["patient_basis"][:, :k])
        cosine[requested_k] = float(np.mean(np.cos(angles)))

    local_tiers = tiers[tiers["compartment"].eq(compartment)]
    borderline = set(local_tiers.loc[local_tiers["borderline_supported"], "CellType"])
    contributions = tiered["contributions"]
    shares = {}
    for pc_name in ("PC1", "PC2"):
        local = contributions[contributions["PC"].eq(pc_name)]
        shares[pc_name] = float(
            local.loc[local["CellType"].isin(borderline), "percent_axis_contribution"].sum() / 100
        )
    expected_share = len(borderline) / int(local_tiers["tiered_primary_keep"].sum())
    dominance_threshold = min(0.50, 2.5 * expected_share)
    geometry_flag = distance_rho < 0.90 or cosine[2] < 0.80
    dominance_flag = max(shares.values()) > dominance_threshold
    recommendation = (
        "REVIEW_STRICT_AS_POSSIBLE_PRIMARY"
        if geometry_flag or dominance_flag
        else "KEEP_TIERED_PRIMARY_WITH_STRICT_QC_SENSITIVITY"
    )
    stability_rows.append(dict(
        compartment=compartment,
        analysis_family=ANALYSIS_FAMILIES[compartment],
        pairwise_distance_spearman=distance_rho,
        mean_cosine_first_2_patient_pc_subspaces=cosine[2],
        mean_cosine_first_5_patient_pc_subspaces=cosine[5],
        borderline_share_pc1=shares["PC1"],
        borderline_share_pc2=shares["PC2"],
        expected_borderline_share=expected_share,
        dominance_review_threshold=dominance_threshold,
        geometry_review_flag=geometry_flag,
        borderline_dominance_flag=dominance_flag,
        recommendation=recommendation,
    ))
stability = pd.DataFrame(stability_rows)


# ------------------------------- Exports ---------------------------------
def collect(key):
    return pd.concat([
        results[compartment][analysis_set][key]
        for compartment in COMPARTMENTS
        for analysis_set in ("tiered_primary", "strict_qc")
    ], ignore_index=True)


patient_scores = collect("patient_scores")
explained = collect("explained")
loadings = collect("loadings")
contributions = collect("contributions")
pc_tests = collect("tests")
outliers = collect("outliers")

atomic_csv(pd.DataFrame(input_counts), OUTPUT_DIR / "pca_input_counts.csv")
atomic_csv(pd.DataFrame(input_hashes), OUTPUT_DIR / "pca_input_hashes.csv")
atomic_csv(stability, OUTPUT_DIR / "pca_tiered_vs_strict_stability.csv")
atomic_csv(explained, OUTPUT_DIR / "pca_explained_variance.csv")
atomic_csv(patient_scores, OUTPUT_DIR / "pca_patient_scores.csv.gz", compression="gzip")
atomic_csv(loadings, OUTPUT_DIR / "pca_feature_loadings.csv.gz", compression="gzip")
atomic_csv(contributions, OUTPUT_DIR / "pca_component_contributions.csv")
atomic_csv(pc_tests, OUTPUT_DIR / "pca_pc_bmi_associations.csv")
atomic_csv(outliers, OUTPUT_DIR / "pca_patient_outlier_review.csv")


# -------------------------- Publication graphics -------------------------
mpl.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})

# Main: tiered primary, colored by continuous BMI.
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.65), constrained_layout=True)
normalizer = mpl.colors.Normalize(patient_scores["bmi"].min(), patient_scores["bmi"].max())
scatter = None
for panel, (axis, compartment) in enumerate(zip(axes, COMPARTMENTS)):
    result = results[compartment]["tiered_primary"]
    local = result["patient_scores"]
    variance = result["pca"].explained_variance_ratio_
    scatter = axis.scatter(
        local["PC1"], local["PC2"], c=local["bmi"], cmap="viridis", norm=normalizer,
        s=23, alpha=0.86, edgecolors="white", linewidths=0.25,
    )
    axis.axhline(0, color="#BBBBBB", linewidth=0.6, zorder=0)
    axis.axvline(0, color="#BBBBBB", linewidth=0.6, zorder=0)
    axis.set_title(DISPLAY_NAMES[compartment])
    axis.set_xlabel(f"PC1 ({100 * variance[0]:.1f}%)")
    axis.set_ylabel(f"PC2 ({100 * variance[1]:.1f}%)")
    axis.text(-0.12, 1.04, chr(65 + panel), transform=axis.transAxes,
              fontsize=12, fontweight="bold")
colorbar = fig.colorbar(scatter, ax=axes, shrink=0.88, pad=0.02)
colorbar.set_label("BMI (kg/m²)")
save_figure(fig, OUTPUT_DIR / "Figure_PCA_tiered_cap5")
plt.show()
plt.close(fig)

# Supplement: tiered versus strict, categorical BMI with 95% data ellipses.
group_order = ("normal_weight", "overweight", "obese")
group_labels = {"normal_weight": "Normal weight", "overweight": "Overweight", "obese": "Obese"}
group_colors = {"normal_weight": "#3B82B8", "overweight": "#E69F00", "obese": "#C44E52"}
fig, axes = plt.subplots(3, 2, figsize=(7.7, 10.0), constrained_layout=True)
for row, compartment in enumerate(COMPARTMENTS):
    for column, analysis_set in enumerate(("tiered_primary", "strict_qc")):
        axis = axes[row, column]
        result = results[compartment][analysis_set]
        local = result["patient_scores"]
        variance = result["pca"].explained_variance_ratio_
        for group_name in group_order:
            group = local[local["bmi_group"].eq(group_name)]
            axis.scatter(group["PC1"], group["PC2"], color=group_colors[group_name],
                         s=20, alpha=0.78, edgecolors="white", linewidths=0.25)
            ellipse = ellipse_xy(group["PC1"], group["PC2"])
            if ellipse is not None:
                axis.plot(*ellipse, color=group_colors[group_name], linewidth=1.0)
        axis.axhline(0, color="#BBBBBB", linewidth=0.6, zorder=0)
        axis.axvline(0, color="#BBBBBB", linewidth=0.6, zorder=0)
        set_label = "Tiered primary" if analysis_set == "tiered_primary" else "Strict QC"
        axis.set_title(f"{DISPLAY_NAMES[compartment]} — {set_label}")
        axis.set_xlabel(f"PC1 ({100 * variance[0]:.1f}%)")
        axis.set_ylabel(f"PC2 ({100 * variance[1]:.1f}%)")
handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=group_colors[group],
                  markeredgecolor="white", label=group_labels[group]) for group in group_order]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.015),
           ncol=3, frameon=False)
save_figure(fig, OUTPUT_DIR / "Figure_PCA_tiered_vs_strict_sensitivity")
plt.show()
plt.close(fig)

# Supplement: cumulative variance.
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4), constrained_layout=True)
for axis, compartment in zip(axes, COMPARTMENTS):
    for analysis_set, color, marker in (
        ("tiered_primary", "#2C7FB8", "o"), ("strict_qc", "#D95F0E", "s")
    ):
        local = explained[
            explained["compartment"].eq(compartment)
            & explained["analysis_set"].eq(analysis_set)
        ]
        axis.plot(np.arange(1, len(local) + 1),
                  100 * local["cumulative_explained_variance_ratio"],
                  color=color, marker=marker, markersize=3.5, linewidth=1.3,
                  label="Tiered primary" if analysis_set == "tiered_primary" else "Strict QC")
    axis.set_title(DISPLAY_NAMES[compartment])
    axis.set_xlabel("Number of PCs")
    axis.set_ylabel("Cumulative variance explained (%)")
    axis.set_xticks(np.arange(1, N_PCS_SAVE + 1))
    axis.grid(axis="y", color="#E6E6E6", linewidth=0.6)
axes[0].legend(frameon=False)
save_figure(fig, OUTPUT_DIR / "Figure_PCA_scree_sensitivity")
plt.show()
plt.close(fig)


# ------------------------------- Report ----------------------------------
report = [
    "# BayesPrism publication PCA and tiered-QC report", "",
    "This cell fitted no Bayesian model and removed no patient automatically.", "",
    f"- Endpoint: `{PRIMARY_VARIANT}`",
    "- Weighting: equal total squared weight per retained correlation cluster within cell type and per cell type.",
    "- Tiered primary excludes only zero-total or median zero-gene-fraction >0.20 components.",
    "- Strict sensitivity additionally excludes low-theta/high-theta-CV components.",
    "- Immune coarse plus non-immune form the broad primary family; immune fine is a nested secondary family.",
    "- PCA is descriptive QC and must not be used to select the version with stronger BMI separation.", "",
    "## Stability decisions", "",
]
for row in stability.itertuples(index=False):
    report.append(
        f"- {DISPLAY_NAMES[row.compartment]}: distance rho={row.pairwise_distance_spearman:.3f}; "
        f"PC1–2 subspace cosine={row.mean_cosine_first_2_patient_pc_subspaces:.3f}; "
        f"`{row.recommendation}`."
    )
report.extend([
    "", "Outlier flags request source/QC inspection only; biological outliers are not automatically excluded.",
    "The theta table contains both original within-compartment q-values and recalculated global q-values.",
])
report_text = "\n".join(report) + "\n"
atomic_text(report_text, OUTPUT_DIR / "PCA_QC_REPORT.md")

pca_manifest = {
    "pca_version": "publication-pca-tiered-qc-v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "output_directory": str(OUTPUT_DIR),
    "combined_review_directory": str(REVIEW_DIR),
    "combined_review_manifest_sha256": file_sha256(manifest_path),
    "primary_variant": PRIMARY_VARIANT,
    "cluster_kind": CLUSTER_KIND,
    "analysis_families": ANALYSIS_FAMILIES,
    "expected": EXPECTED,
    "notes": [
        "No Bayesian model fitted.", "No patient automatically excluded.",
        "Features were standardized across the same 127 patients before PCA.",
        "PCA weights prevent signature-rich or redundant blocks dominating by feature count.",
    ],
}
atomic_text(json.dumps(pca_manifest, indent=2), OUTPUT_DIR / "pca_manifest.json")

display(Markdown(report_text))
print("\nCOMPONENT TIERS")
display(tiers[["compartment", "CellType", "analysis_tier", "tiered_primary_keep",
               "strict_qc_keep", "review_reasons"]].sort_values(
                   ["compartment", "analysis_tier", "CellType"]
               ))
print("\nPCA INPUT COUNTS")
display(pd.DataFrame(input_counts))
print("\nTIERED VERSUS STRICT STABILITY")
display(stability)
print("\nPC–BMI ASSOCIATIONS (HC3; descriptive)")
display(pc_tests.sort_values(["compartment", "analysis_set", "p_value"]))
print("\nPATIENT OUTLIER FLAGS FOR REVIEW ONLY")
display(outliers[outliers["review_flag"]].sort_values(
    ["compartment", "analysis_set", "squared_distance"], ascending=[True, True, False]
))
print("\nTHETA–BMI TOP RAW P-VALUES WITH GLOBAL MULTIPLICITY")
display(theta_bmi.nsmallest(10, "p_value")[[
    "compartment", "component", "beta", "p_value", "q_value",
    "q_global_all_components", "q_global_modeled_components",
]])
print("\nSaved publication PCA outputs to:", OUTPUT_DIR)

